In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from sklearn.cluster import KMeans
import folium
import numpy as np
import warnings

In [36]:
# Suprimir avisos de convergência do KMeans, que podem ser comuns em alguns datasets.
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


In [37]:
# --- 1. Carregar os Dados da Planilha ---

# NOME DO ARQUIVO DA PLANILHA
file_path = '/Users/igorbione/Documents/Projeto_Logistica_TT/data/raw/TT Prospec.xlsx'
nome_sheet = 'Planilha3'

# TENTE ADAPTAR OS NOMES DAS COLUNAS AQUI SE ESTIVEREM DIFERENTES NA SUA PLANILHA
# Coluna que contém o endereço completo para geocodificação
address_column_name = 'ADDRESS' # <--- AJUSTE AQUI SE O NOME FOR DIFERENTE

# Coluna que contém o volume de pacotes (para a Etapa 2, mas bom já ter)
volume_column_name = 'VOLUME TIKTOK PICKUP' # <--- AJUSTE AQUI SE O NOME FOR DIFERENTE

# Coluna que pode servir como identificador único do seller (opcional, usaremos o índice se não tiver)
seller_id_column_name = 'SELLER ID' # <--- AJUSTE AQUI SE O NOME FOR DIFERENTE OU DEIXE EM BRANCO SE NÃO TIVER


In [121]:
try:
    df = pd.read_excel(file_path, sheet_name=nome_sheet)
    print(f"Planilha '{file_path}' carregada com sucesso!")
    print("Primeiras 5 linhas do DataFrame:")
    print(df.head())
    print("\nInformações das colunas:")
    df.info()
    print("-" * 30)

    # Renomear colunas para padronizar, caso os nomes da sua planilha sejam diferentes
    # Ajuste 'Seu Nome da Coluna de Endereço' para o nome exato da coluna na sua planilha
    # E 'Seu Nome da Coluna de Volume' para o nome exato da coluna de volume
    # Se os nomes já forem 'Endereço Completo' e 'Volume Pacotes', pode ignorar ou remover essas linhas.
    # Exemplo: df.rename(columns={'NOME DA COLUNA ENDERECO NA SUA PLANILHA': address_column_name,
    #                              'NOME DA COLUNA VOLUME NA SUA PLANILHA': volume_column_name}, inplace=True)

    # Verificar se as colunas essenciais existem
    if address_column_name not in df.columns:
        raise ValueError(f"Coluna '{address_column_name}' não encontrada na planilha. Por favor, ajuste 'address_column_name'.")
    if volume_column_name not in df.columns:
        print(f"Atenção: Coluna '{volume_column_name}' não encontrada. A Etapa 2 pode precisar de ajuste.")
        df[volume_column_name] = 0 # Adiciona coluna de volume com zeros se não existir, para evitar erro futuro

    # Se a coluna de seller_id não for encontrada ou definida, cria uma usando o índice
    if seller_id_column_name not in df.columns or not seller_id_column_name:
        df['seller_id'] = df.index + 1 # Criar um ID sequencial
        print("Coluna 'seller_id' criada a partir do índice.")
        seller_id_column_name = 'seller_id'
    else:
        # Renomeia para 'seller_id' para padronizar o uso no código
        df.rename(columns={seller_id_column_name: 'seller_id'}, inplace=True)
        seller_id_column_name = 'seller_id'


except FileNotFoundError:
    print(f"Erro: O arquivo '{file_path}' não foi encontrado. Certifique-se de que o arquivo está no mesmo diretório do seu script ou forneça o caminho completo.")
    exit()
except Exception as e:
    print(f"Ocorreu um erro ao carregar ou processar a planilha: {e}")
    exit()


Planilha '/Users/igorbione/Documents/Projeto_Logistica_TT/data/raw/TT Prospec.xlsx' carregada com sucesso!
Primeiras 5 linhas do DataFrame:
  EXPECT PICKUP CLIENTE            SELLER ID  \
0    2025-05-26  TIKTOK  7496179179138159616   
1    2025-05-26  TIKTOK  7496161546569480192   
2    2025-05-26  TIKTOK  7496147585319799808   
3    2025-05-26  TIKTOK  7496162496353829888   
4    2025-05-26  TIKTOK  7496186178403340288   

                                             ADDRESS       CITY      STATE  \
0                         RUA CAPITÃO FERRAIUOLO 367  São Paulo  São Paulo   
1                    RUA DOUTOR ALCIDES DE CAMPOS 35  São Paulo  São Paulo   
2  RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...  São Paulo  São Paulo   
3                               RUA JOÃO BOEMER 1259  São Paulo  São Paulo   
4                    AVENIDA JOÃO DIAS 2319 BOX 1210  São Paulo  São Paulo   

    ZIP CODE            AREA                ROTA  VOLUME TIKTOK PICKUP  
0  03348-000  Vila Invernada 

In [6]:
df

,EXPECT PICKUP,CLIENTE,seller_id,ADDRESS,CITY,STATE,ZIP CODE,AREA,ROTA,VOLUME TIKTOK PICKUP
0,2025-05-26,TIKTOK,7496179179138159616,RUA CAPITÃO FERRAIUOLO 367,São Paulo,São Paulo,03348-000,Vila Invernada,SP Capital - Leste,3
1,2025-05-26,TIKTOK,7496161546569480192,RUA DOUTOR ALCIDES DE CAMPOS 35,São Paulo,São Paulo,04336-160,Americanópolis,SP Capital - Sul,2
2,2025-05-26,TIKTOK,7496147585319799808,RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...,São Paulo,São Paulo,03018-010,Brás,SP Capital - Brás,9
3,2025-05-26,TIKTOK,7496162496353829888,RUA JOÃO BOEMER 1259,São Paulo,São Paulo,03018-000,Brás,SP Capital - Brás,2
4,2025-05-26,TIKTOK,7496186178403340288,AVENIDA JOÃO DIAS 2319 BOX 1210,São Paulo,São Paulo,04723-003,Santo Amaro,SP Capital - Sul,1
...,...,...,...,...,...,...,...,...,...,...
265,2025-05-26,TIKTOK,7496167854168179712,RUA MILLER 614,São Paulo,São Paulo,03011-011,Brás,SP Capital - Brás,4
266,2025-05-26,TIKTOK,7496183141361420288,RUA MILLER 614,São Paulo,São Paulo,03011-011,Brás,SP Capital - Brás,1
267,2025-05-26,TIKTOK,7496182107474329600,RUA PADRE POMPEU DE ALMEIDA 230,São Paulo,São Paulo,08331-040,Cidade Satélite Santa Bárbara,SP Capital - Leste,1
268,2025-05-26,TIKTOK,7496163512946820096,RUA BOM PASTOR 2795,São Paulo,São Paulo,04203-000,Ipiranga,SP Capital - Sul,246


In [7]:
# --- 2. Configuração do Geocodificador ---

print("Configurando geocodificador Nominatim...")
geolocator = Nominatim(user_agent="meu_app_logistica_analise_sellers_v2")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1) # Atraso de 1 segundo entre requisições


Configurando geocodificador Nominatim...


In [15]:
# --- 3. Geocodificação dos Endereços ---

print("Iniciando geocodificação dos endereços. Isso pode levar um tempo, dependendo do número de linhas...")

# Aplica a função de geocodificação para cada endereço na coluna definida
# Tratamento de valores NaN (Not a Number) na coluna de endereço antes de aplicar o geocode
df['location'] = df[address_column_name].astype(str).apply(geocode)

# Extrai latitude e longitude
df['latitude'] = df['location'].apply(lambda loc: loc.latitude if loc else None)
df['longitude'] = df['location'].apply(lambda loc: loc.longitude if loc else None)

# Remove a coluna 'location' temporária
df = df.drop(columns=['location'])

print("Geocodificação concluída!")
print("DataFrame com coordenadas geográficas (primeiras 5 linhas):")
print(df.head())

# Remover linhas onde a geocodificação falhou (latitude ou longitude são nulas)
initial_rows = len(df)
df.dropna(subset=['latitude', 'longitude'], inplace=True)
rows_after_dropna = len(df)
print(f"\n{initial_rows - rows_after_dropna} linhas foram removidas devido a falha na geocodificação.")
print(f"Total de {rows_after_dropna} sellers com coordenadas válidas para clusterização.")
print("-" * 30)

# Salvando os resultados geocodificados
output_geocoded_file = 'sellers_geocodificados.csv'
df.to_csv(output_geocoded_file, index=False)
print(f"Dados geocodificados salvos em '{output_geocoded_file}'")

# Verificar se há dados suficientes após a remoção de nulos para continuar
if df.empty:
    print("Não há dados de latitude/longitude válidos para clusterizar e plotar. Verifique o arquivo de entrada e os endereços.")
    exit()


Iniciando geocodificação dos endereços. Isso pode levar um tempo, dependendo do número de linhas...


RateLimiter caught an error, retrying (0/2 tries). Called with (*('RUA BORGES DE FIGUEIREDO 1133 RUA BORGES DE FIGUEIREDO, 1133',), **{}).
Traceback (most recent call last):
  File "/Users/igorbione/anaconda3/lib/python3.11/site-packages/urllib3/connectionpool.py", line 466, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "/Users/igorbione/anaconda3/lib/python3.11/site-packages/urllib3/connectionpool.py", line 461, in _make_request
    httplib_response = conn.getresponse()
                       ^^^^^^^^^^^^^^^^^^
  File "/Users/igorbione/anaconda3/lib/python3.11/http/client.py", line 1395, in getresponse
    response.begin()
  File "/Users/igorbione/anaconda3/lib/python3.11/http/client.py", line 325, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/Users/igorbione/anaconda3/lib/python3.11/http/client.py", line 286, in _read_status
    line = str(self.fp.readline(_MAXLINE +

KeyboardInterrupt: 

In [16]:
function getLatLngByZipcode(zipcode) 
{
    var geocoder = new google.maps.Geocoder();
    var address = zipcode;
    geocoder.geocode({ 'address': address }, function (results, status) {
        if (status == google.maps.GeocoderStatus.OK) {
            var latitude = results[0].geometry.location.lat();
            var longitude = results[0].geometry.location.lng();
            alert("Latitude: " + latitude + "\nLongitude: " + longitude);
        } else {
            alert("Request failed.")
        }
    });
    return [latitude, longitude];
}

SyntaxError: invalid syntax (1582227981.py, line 1)

In [34]:
import pgeocode

nomi = pgeocode.Nominatim('BR')
query = nomi.query_postal_code("03377-000")



data = {
    "lat": query["latitude"],
    "lon": query["longitude"]
}

print(data)

{'lat': nan, 'lon': nan}


In [26]:
nomi = pgeocode.Nominatim('BR')

In [28]:
nomi.query_location("Recife", top_k=3)

,country_code,postal_code,place_name,state_name,state_code,county_name,county_code,community_name,community_code,latitude,longitude,accuracy
5332,BR,50000-000,Recife,Pernambuco,30,Recife,2611606.0,NaN,NaN,-8.0117,-34.9529,2


In [31]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="geolocalização")
location = geolocator.geocode("Rua Doutor Seng, Bela Vista, São Paulo - SP")

print((location.latitude, location.longitude))


(-23.5614229, -46.65035)


In [9]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

def geocode_addresses(df):
    """
    Adiciona colunas de latitude e longitude ao dataframe baseado na coluna ADDRESS
    """
    # Inicializa o geocodificador
    geolocator = Nominatim(user_agent="geolocalizacao")
    
    # Cria listas para armazenar os resultados
    latitudes = []
    longitudes = []
    
    # Percorre cada endereço na coluna ADDRESS
    for index, address in df['ADDRESS'].items():
        try:
            print(f"Processando linha {index}: {address}")
            
            # Geocodifica o endereço
            location = geolocator.geocode(address)
            
            if location:
                # Se encontrou localização, adiciona latitude e longitude
                latitudes.append(location.latitude)
                longitudes.append(location.longitude)
                print(f"  ✓ Encontrado: {location.latitude}, {location.longitude}")
            else:
                # Se não encontrou, adiciona valores nulos
                latitudes.append(None)
                longitudes.append(None)
                print(f"  ✗ Endereço não encontrado")
                
        except Exception as e:
            # Em caso de erro, adiciona valores nulos
            latitudes.append(None)
            longitudes.append(None)
            print(f"  ✗ Erro: {e}")
        
        # Pausa para respeitar os limites da API (recomendado)
        time.sleep(1)
    
    # Adiciona as novas colunas ao dataframe
    df['LATITUDE'] = latitudes
    df['LONGITUDE'] = longitudes
    
    return df

# Exemplo de uso:
df_with_coordinates = geocode_addresses(df)

# Se quiser salvar o resultado em um novo arquivo CSV:
# df_with_coordinates.to_csv('enderecos_com_coordenadas.csv', index=False)

# Para verificar os resultados:
# print(df_with_coordinates[['ADDRESS', 'LATITUDE', 'LONGITUDE']].head())

Processando linha 0: RUA CAPITÃO FERRAIUOLO 367
  ✓ Encontrado: -23.5652896, -46.5629871
Processando linha 1: RUA DOUTOR ALCIDES DE CAMPOS 35
  ✓ Encontrado: -23.6674386, -46.6453186
Processando linha 2: RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERDO DA COLEGIO
  ✗ Endereço não encontrado
Processando linha 3: RUA JOÃO BOEMER 1259
  ✓ Encontrado: -23.5303995, -46.6143002
Processando linha 4: AVENIDA JOÃO DIAS 2319 BOX 1210
  ✗ Erro: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=AVENIDA+JO%C3%83O+DIAS+2319+BOX+1210&format=json&limit=1 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=1)"))
Processando linha 5: RUA ORIENTE 655 7 ANDAR
  ✗ Erro: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=RUA+ORIENTE+655+7+ANDAR&format=json&limit=1 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='nomi

In [10]:
# Se quiser salvar o resultado em um novo arquivo CSV:
df_with_coordinates.to_csv('enderecos_com_coordenadas.csv', index=False)

# Para verificar os resultados:
print(df_with_coordinates[['ADDRESS', 'LATITUDE', 'LONGITUDE']].head())

                                             ADDRESS   LATITUDE  LONGITUDE
0                         RUA CAPITÃO FERRAIUOLO 367 -23.565290 -46.562987
1                    RUA DOUTOR ALCIDES DE CAMPOS 35 -23.667439 -46.645319
2  RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...        NaN        NaN
3                               RUA JOÃO BOEMER 1259 -23.530400 -46.614300
4                    AVENIDA JOÃO DIAS 2319 BOX 1210        NaN        NaN


In [13]:
df_2 = df_with_coordinates[df_with_coordinates['LATITUDE'].isna()]

In [14]:
df_with_coordinates2 = geocode_addresses(df_2)

Processando linha 2: RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERDO DA COLEGIO
  ✗ Endereço não encontrado
Processando linha 4: AVENIDA JOÃO DIAS 2319 BOX 1210
  ✓ Encontrado: -23.6449312, -46.717928
Processando linha 5: RUA ORIENTE 655 7 ANDAR
  ✗ Endereço não encontrado
Processando linha 7: AVENIDA M BOI MIRIM 2089 SALA COMERCIAL
  ✗ Endereço não encontrado
Processando linha 9: RUA IOSOSUKE OKAUE 762 LOJA
  ✗ Endereço não encontrado
Processando linha 10: AVENIDA RENATA 693 SALAO COMERCIAL AZUL
  ✗ Endereço não encontrado
Processando linha 11: RUA BORGES DE FIGUEIREDO 1133 RUA BORGES DE FIGUEIREDO, 1133
  ✗ Endereço não encontrado
Processando linha 12: RUA MANUEL RAMOS PAIVA 506 ANDAR 2 SALA 01
  ✗ Endereço não encontrado
Processando linha 13: RUA SANTA ISABEL 160 CONJUNTO 25
  ✗ Endereço não encontrado
Processando linha 14: RUA JULIA TREVISANI GANNAM 256
  ✓ Encontrado: -23.4957259, -46.6653765
Processando linha 16: RUA CHICO PONTES 1512 GALPAO 88
  ✗ Endereço não encontrado
Process

/var/folders/vt/21jnptwn0s7_2xg33qtnlzyw0000gn/T/ipykernel_17723/2375002336.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['LATITUDE'] = latitudes
/var/folders/vt/21jnptwn0s7_2xg33qtnlzyw0000gn/T/ipykernel_17723/2375002336.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['LONGITUDE'] = longitudes


In [16]:
df_with_coordinates2[df_with_coordinates2['LATITUDE'].isna()]

,EXPECT PICKUP,CLIENTE,seller_id,ADDRESS,CITY,STATE,ZIP CODE,AREA,ROTA,VOLUME TIKTOK PICKUP,LATITUDE,LONGITUDE
2,2025-05-26,TIKTOK,7496147585319799808,RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...,São Paulo,São Paulo,03018-010,Brás,SP Capital - Brás,9,NaN,NaN
5,2025-05-26,TIKTOK,7496162359001320448,RUA ORIENTE 655 7 ANDAR,São Paulo,São Paulo,03016-001,Brás,SP Capital - Brás,5,NaN,NaN
7,2025-05-26,TIKTOK,7496183052168170496,AVENIDA M BOI MIRIM 2089 SALA COMERCIAL,São Paulo,São Paulo,04905-022,Jardim das Flores,SP Capital - Sul,2,NaN,NaN
9,2025-05-26,TIKTOK,7496182910325980160,RUA IOSOSUKE OKAUE 762 LOJA,São Paulo,São Paulo,08265-150,Jardim Helian,SP Capital - Leste,1,NaN,NaN
10,2025-05-26,TIKTOK,7496181401129490432,AVENIDA RENATA 693 SALAO COMERCIAL AZUL,São Paulo,São Paulo,03377-000,Chácara Belenzinho,SP Capital - Leste,2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
257,2025-05-26,TIKTOK,7496180072362579968,RUA CARNOT 70 ANDAR 3 SALA 41,São Paulo,São Paulo,03032-030,Canindé,SP Capital - Brás,5,NaN,NaN
258,2025-05-26,TIKTOK,7496180071308040192,RUA CARNOT 70 3-41,São Paulo,São Paulo,03032-030,Canindé,SP Capital - Brás,1,NaN,NaN
259,2025-05-26,TIKTOK,7496168616964620288,RUA DOUTOR SENG 182 APT 313B,São Paulo,São Paulo,01331-020,Bela Vista,SP Capital - Centro,1,NaN,NaN
261,2025-05-26,TIKTOK,7496163579931360256,RUA RIO BONITO 1053 LOJA,São Paulo,São Paulo,03023-000,Brás,SP Capital - Brás,1,NaN,NaN


In [22]:
df_with_coordinates2

,EXPECT PICKUP,CLIENTE,seller_id,ADDRESS,CITY,STATE,ZIP CODE,AREA,ROTA,VOLUME TIKTOK PICKUP,LATITUDE,LONGITUDE
2,2025-05-26,TIKTOK,7496147585319799808,RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...,São Paulo,São Paulo,03018-010,Brás,SP Capital - Brás,9,NaN,NaN
4,2025-05-26,TIKTOK,7496186178403340288,AVENIDA JOÃO DIAS 2319 BOX 1210,São Paulo,São Paulo,04723-003,Santo Amaro,SP Capital - Sul,1,-23.644931,-46.717928
5,2025-05-26,TIKTOK,7496162359001320448,RUA ORIENTE 655 7 ANDAR,São Paulo,São Paulo,03016-001,Brás,SP Capital - Brás,5,NaN,NaN
7,2025-05-26,TIKTOK,7496183052168170496,AVENIDA M BOI MIRIM 2089 SALA COMERCIAL,São Paulo,São Paulo,04905-022,Jardim das Flores,SP Capital - Sul,2,NaN,NaN
9,2025-05-26,TIKTOK,7496182910325980160,RUA IOSOSUKE OKAUE 762 LOJA,São Paulo,São Paulo,08265-150,Jardim Helian,SP Capital - Leste,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
257,2025-05-26,TIKTOK,7496180072362579968,RUA CARNOT 70 ANDAR 3 SALA 41,São Paulo,São Paulo,03032-030,Canindé,SP Capital - Brás,5,NaN,NaN
258,2025-05-26,TIKTOK,7496180071308040192,RUA CARNOT 70 3-41,São Paulo,São Paulo,03032-030,Canindé,SP Capital - Brás,1,NaN,NaN
259,2025-05-26,TIKTOK,7496168616964620288,RUA DOUTOR SENG 182 APT 313B,São Paulo,São Paulo,01331-020,Bela Vista,SP Capital - Centro,1,NaN,NaN
261,2025-05-26,TIKTOK,7496163579931360256,RUA RIO BONITO 1053 LOJA,São Paulo,São Paulo,03023-000,Brás,SP Capital - Brás,1,NaN,NaN


In [24]:
from geopy.geocoders import Nominatim

def get_lat_long_from_cep(cep):
    geolocator = Nominatim(user_agent="geoapiExercises")
    location = geolocator.geocode(f"{cep}, São Paulo, Brasil")
    if location:
        return (location.latitude, location.longitude)
    else:
        return None

# Exemplo de uso:
cep = "01001-000"  # CEP exemplo (Centro de São Paulo)
coordenadas = get_lat_long_from_cep(cep)

if coordenadas:
    print(f"Latitude: {coordenadas[0]}, Longitude: {coordenadas[1]}")
else:
    print("CEP não encontrado ou serviço indisponível.")

GeocoderInsufficientPrivileges: Non-successful status code 403

In [26]:
import requests

def consultar_cep_viacep(cep):
    # Remove caracteres não numéricos do CEP
    cep = ''.join(filter(str.isdigit, cep))
    if len(cep) != 8:
        return {"erro": "CEP inválido. Deve conter 8 dígitos."}

    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        response = requests.get(url)
        response.raise_for_status() # Lança um erro para status de erro HTTP
        data = response.json()
        if "erro" in data and data["erro"] == True:
            return {"erro": "CEP não encontrado."}
        return data
    except requests.exceptions.RequestException as e:
        return {"erro": f"Erro na requisição: {e}"}

# Exemplo de uso para um CEP de São Paulo
cep_sp = "03018-010" # Praça da Sé, São Paulo, SP
endereco = consultar_cep_viacep(cep_sp)

if "erro" in endereco:
    print(f"Erro: {endereco['erro']}")
else:
    print(f"CEP: {endereco.get('cep')}")
    print(f"Logradouro: {endereco.get('logradouro')}")
    print(f"Bairro: {endereco.get('bairro')}")
    print(f"Cidade: {endereco.get('localidade')}")
    print(f"Estado: {endereco.get('uf')}")

cep_inexistente = "99999999"
endereco_inexistente = consultar_cep_viacep(cep_inexistente)
print("\n", endereco_inexistente)

CEP: 03018-010
Logradouro: Rua Coronel Emídio Piedade
Bairro: Brás
Cidade: São Paulo
Estado: SP

 {'erro': 'true'}


In [33]:
import requests

def obter_endereco_string_unica(cep):
    # Remove caracteres não numéricos do CEP
    cep = ''.join(filter(str.isdigit, cep))
    if len(cep) != 8:
        return "Erro: CEP inválido. Deve conter 8 dígitos."

    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Lança um erro para status de erro HTTP
        data = response.json()

        if "erro" in data and data["erro"] == True:
            return "Erro: CEP não encontrado."
        
        # Monta a string única do endereço
        logradouro = data.get('logradouro', 'Não informado')
        bairro = data.get('bairro', 'Não informado')
        localidade = data.get('localidade', 'Não informada')
        uf = data.get('uf', 'NI')
        cep_formatado = data.get('cep', 'Não informado')

        # Exemplo de formatação: "Rua Exemplo, Bairro X, Cidade Y - Estado Z, CEP: 00000-000"
        endereco_completo = (
            f"{logradouro}, {bairro}, {localidade} - {uf}"
        )
        return endereco_completo

    except requests.exceptions.RequestException as e:
        return f"Erro na requisição: {e}"

# --- Exemplos de uso para CEPs de São Paulo ---

# CEP da Praça da Sé, São Paulo, SP
cep_sp = "01331-020" 
string_endereco_sp = obter_endereco_string_unica(cep_sp)
print(f"Endereço para o CEP {cep_sp}: {string_endereco_sp}")

print("-" * 50)

# CEP de uma rua na Vila Olímpia, São Paulo, SP
cep_vila_olimpia = "04547-000"
string_endereco_vila_olimpia = obter_endereco_string_unica(cep_vila_olimpia)
print(f"Endereço para o CEP {cep_vila_olimpia}: {string_endereco_vila_olimpia}")

print("-" * 50)

# Exemplo de CEP inexistente
cep_inexistente = "99999999"
string_endereco_inexistente = obter_endereco_string_unica(cep_inexistente)
print(f"Endereço para o CEP {cep_inexistente}: {string_endereco_inexistente}")

Endereço para o CEP 01331-020: Rua Doutor Seng, Bela Vista, São Paulo - SP
--------------------------------------------------
Endereço para o CEP 04547-000: Rua Gomes de Carvalho, Vila Olímpia, São Paulo - SP
--------------------------------------------------
Endereço para o CEP 99999999: Não informado, Não informado, Não informada - NI


In [34]:
string_endereco_sp

'Rua Doutor Seng, Bela Vista, São Paulo - SP'

In [42]:
import pandas as pd
from geopy.geocoders import Nominatim
import time
import requests
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

def obter_endereco_string_unica(cep):
    """
    Obtém endereço completo a partir do CEP usando a API ViaCEP
    """
    # Remove caracteres não numéricos do CEP
    cep = ''.join(filter(str.isdigit, cep))
    if len(cep) != 8:
        return "Erro: CEP inválido. Deve conter 8 dígitos."
    
    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Lança um erro para status de erro HTTP
        data = response.json()
        
        if "erro" in data and data["erro"] == True:
            return "Erro: CEP não encontrado."
        
        # Monta a string única do endereço
        logradouro = data.get('logradouro', 'Não informado')
        bairro = data.get('bairro', 'Não informado')
        localidade = data.get('localidade', 'Não informada')
        uf = data.get('uf', 'NI')
        
        # Exemplo de formatação: "Rua Exemplo, Bairro X, Cidade Y - Estado Z"
        endereco_completo = f"{logradouro}, {bairro}, {localidade} - {uf}"
        return endereco_completo
        
    except requests.exceptions.RequestException as e:
        return f"Erro na requisição: {e}"

def adicionar_endereco_completo(df):
    """
    Adiciona coluna ENDERECO_COMPLETO baseada na coluna ZIPCODE
    """
    print("Criando coluna ENDERECO_COMPLETO a partir dos CEPs...")
    enderecos_completos = []
    
    for index, zipcode in df['ZIP CODE'].items():
        print(f"Processando CEP linha {index}: {zipcode}")
        endereco = obter_endereco_string_unica(str(zipcode))
        enderecos_completos.append(endereco)
        print(f"  → {endereco}")
        
        # Pausa pequena para não sobrecarregar a API ViaCEP
        time.sleep(0.5)
    
    df['ENDERECO_COMPLETO'] = enderecos_completos
    return df

def geocode_addresses(df):
    """
    Adiciona colunas de latitude e longitude ao dataframe baseado na coluna ENDERECO_COMPLETO
    """
    # Inicializa o geocodificador com timeout personalizado
    geolocator = Nominatim(
        user_agent="geolocalizacao_app_v1.0",
        timeout=10  # Timeout de 10 segundos
    )
    
    # Cria listas para armazenar os resultados
    latitudes = []
    longitudes = []
    
    print("\nIniciando geocodificação dos endereços completos...")
    
    # Percorre cada endereço na coluna ENDERECO_COMPLETO
    for index, endereco_completo in df['ENDERECO_COMPLETO'].items():
        # Pula endereços com erro
        if str(endereco_completo).startswith("Erro:"):
            print(f"Pulando linha {index} (erro no CEP): {endereco_completo}")
            latitudes.append(None)
            longitudes.append(None)
            continue
            
        max_retries = 3
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            try:
                print(f"Geocodificando linha {index}: {endereco_completo} (tentativa {retry_count + 1})")
                
                # Geocodifica o endereço
                location = geolocator.geocode(endereco_completo, timeout=10)
                
                if location:
                    # Se encontrou localização, adiciona latitude e longitude
                    latitudes.append(location.latitude)
                    longitudes.append(location.longitude)
                    print(f"  ✓ Encontrado: {location.latitude}, {location.longitude}")
                    success = True
                else:
                    # Se não encontrou, adiciona valores nulos
                    latitudes.append(None)
                    longitudes.append(None)
                    print(f"  ✗ Endereço não encontrado")
                    success = True
                    
            except (GeocoderTimedOut, GeocoderServiceError) as e:
                retry_count += 1
                print(f"  ⚠ Erro de conexão (tentativa {retry_count}): {e}")
                
                if retry_count < max_retries:
                    wait_time = 2 ** retry_count  # Backoff exponencial
                    print(f"  ⏳ Aguardando {wait_time} segundos antes de tentar novamente...")
                    time.sleep(wait_time)
                else:
                    # Todas as tentativas falharam
                    latitudes.append(None)
                    longitudes.append(None)
                    print(f"  ✗ Falha após {max_retries} tentativas")
                    
            except Exception as e:
                # Outros erros não relacionados à conexão
                latitudes.append(None)
                longitudes.append(None)
                print(f"  ✗ Erro: {e}")
                success = True
        
        # Pausa para respeitar os limites da API (aumentada para maior estabilidade)
        time.sleep(2)
    
    # Adiciona as novas colunas ao dataframe
    df['LATITUDE'] = latitudes
    df['LONGITUDE'] = longitudes
    
    return df

def processar_geocodificacao_completa(df):
    """
    Função principal que executa todo o processo:
    1. Cria coluna ENDERECO_COMPLETO a partir de ZIPCODE
    2. Geocodifica usando ENDERECO_COMPLETO
    """
    print("=== INICIANDO PROCESSO DE GEOCODIFICAÇÃO COMPLETA ===\n")
    
    # Passo 1: Criar endereços completos a partir dos CEPs
    df_com_endereco = adicionar_endereco_completo(df.copy())
    
    # Mostrar estatísticas dos endereços obtidos
    enderecos_validos = df_com_endereco['ENDERECO_COMPLETO'].apply(
        lambda x: not str(x).startswith("Erro:")
    ).sum()
    total_enderecos = len(df_com_endereco)
    
    print(f"\n📊 Estatísticas dos CEPs:")
    print(f"   Total de registros: {total_enderecos}")
    print(f"   Endereços válidos obtidos: {enderecos_validos}")
    print(f"   Taxa de sucesso CEPs: {enderecos_validos/total_enderecos*100:.1f}%")
    
    # Passo 2: Geocodificar usando os endereços completos
    df_final = geocode_addresses(df_com_endereco)
    
    # Mostrar estatísticas finais
    coordenadas_validas = df_final['LATITUDE'].notna().sum()
    print(f"\n📊 Estatísticas finais:")
    print(f"   Coordenadas obtidas: {coordenadas_validas}")
    print(f"   Taxa de sucesso geocodificação: {coordenadas_validas/total_enderecos*100:.1f}%")
    
    return df_final

# Exemplo de uso:
df_resultado = processar_geocodificacao_completa(df)

# Para salvar o resultado:
# df_resultado.to_csv('enderecos_geocodificados.csv', index=False)

# Para verificar os resultados:
# print(df_resultado[['ZIPCODE', 'ENDERECO_COMPLETO', 'LATITUDE', 'LONGITUDE']].head(10))

=== INICIANDO PROCESSO DE GEOCODIFICAÇÃO COMPLETA ===

Criando coluna ENDERECO_COMPLETO a partir dos CEPs...
Processando CEP linha 0: 03348-000
  → Rua Capitão Ferraiuolo, Vila Invernada, São Paulo - SP
Processando CEP linha 1: 04336-160
  → Rua Doutor Alcides de Campos, Americanópolis, São Paulo - SP
Processando CEP linha 2: 03018-010
  → Rua Coronel Emídio Piedade, Brás, São Paulo - SP
Processando CEP linha 3: 03018-000
  → Rua João Boemer, Brás, São Paulo - SP
Processando CEP linha 4: 04723-003
  → Avenida João Dias, Santo Amaro, São Paulo - SP
Processando CEP linha 5: 03016-001
  → Rua Oriente, Brás, São Paulo - SP
Processando CEP linha 6: 02415-000
  → Avenida Santa Inês, Parque Mandaqui, São Paulo - SP
Processando CEP linha 7: 04905-022
  → Estrada do M'Boi Mirim, Jardim das Flores, São Paulo - SP
Processando CEP linha 8: 03054-020
  → Rua Coronel Albino Bairão, Belenzinho, São Paulo - SP
Processando CEP linha 9: 08265-150
  → Rua Iososuke Okaue, Jardim Helian, São Paulo - SP
Pro

In [41]:
df.columns

Index(['EXPECT PICKUP', 'CLIENTE', 'seller_id', 'ADDRESS', 'CITY', 'STATE',
       'ZIP CODE', 'AREA', 'ROTA', 'VOLUME TIKTOK PICKUP'],
      dtype='object')

In [46]:
df_resultado.to_excel('enderecos_geocodificados.xlsx')

In [ ]:
df_resultado['LATITUDE']

,EXPECT PICKUP,CLIENTE,seller_id,ADDRESS,CITY,STATE,ZIP CODE,AREA,ROTA,VOLUME TIKTOK PICKUP,ENDERECO_COMPLETO,LATITUDE,LONGITUDE
0,2025-05-26,TIKTOK,7496179179138159616,RUA CAPITÃO FERRAIUOLO 367,São Paulo,São Paulo,03348-000,Vila Invernada,SP Capital - Leste,3,"Rua Capitão Ferraiuolo, Vila Invernada, São Pa...",NaN,NaN
1,2025-05-26,TIKTOK,7496161546569480192,RUA DOUTOR ALCIDES DE CAMPOS 35,São Paulo,São Paulo,04336-160,Americanópolis,SP Capital - Sul,2,"Rua Doutor Alcides de Campos, Americanópolis, ...",NaN,NaN
2,2025-05-26,TIKTOK,7496147585319799808,RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...,São Paulo,São Paulo,03018-010,Brás,SP Capital - Brás,9,"Rua Coronel Emídio Piedade, Brás, São Paulo - SP",NaN,NaN
3,2025-05-26,TIKTOK,7496162496353829888,RUA JOÃO BOEMER 1259,São Paulo,São Paulo,03018-000,Brás,SP Capital - Brás,2,"Rua João Boemer, Brás, São Paulo - SP",NaN,NaN
4,2025-05-26,TIKTOK,7496186178403340288,AVENIDA JOÃO DIAS 2319 BOX 1210,São Paulo,São Paulo,04723-003,Santo Amaro,SP Capital - Sul,1,"Avenida João Dias, Santo Amaro, São Paulo - SP",-23.646264,-46.705345
...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,2025-05-26,TIKTOK,7496167854168179712,RUA MILLER 614,São Paulo,São Paulo,03011-011,Brás,SP Capital - Brás,4,"Rua Miller, Brás, São Paulo - SP",-23.536676,-46.617724
266,2025-05-26,TIKTOK,7496183141361420288,RUA MILLER 614,São Paulo,São Paulo,03011-011,Brás,SP Capital - Brás,1,"Rua Miller, Brás, São Paulo - SP",-23.536676,-46.617724
267,2025-05-26,TIKTOK,7496182107474329600,RUA PADRE POMPEU DE ALMEIDA 230,São Paulo,São Paulo,08331-040,Cidade Satélite Santa Bárbara,SP Capital - Leste,1,"Rua Padre Pompeu de Almeida, Cidade Satélite S...",NaN,NaN
268,2025-05-26,TIKTOK,7496163512946820096,RUA BOM PASTOR 2795,São Paulo,São Paulo,04203-000,Ipiranga,SP Capital - Sul,246,"Rua Bom Pastor, Ipiranga, São Paulo - SP",-23.587125,-46.607393


In [52]:
'Rua Padre Pompeu de Almeida, Cidade Satélite Santa Bárbara, São Paulo - SP'

from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="geolocalização")
location = geolocator.geocode("Rua Padre Pompeu de Almeida, Cidade Satélite Santa Bárbara, São Paulo - SP")

print((location.latitude, location.longitude))

AttributeError: 'NoneType' object has no attribute 'latitude'

In [51]:
geolocator.geocode("Rua Padre Pompeu de Almeida, Cidade Satélite Santa Bárbara, São Paulo - SP")

In [54]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geolocalização")
location = geolocator.geocode("Rua Padre Pompeu de Almeida, São Paulo - SP")

if location:  # Verifica se o endereço foi encontrado
    print((location.latitude, location.longitude))
else:
    print("Endereço não encontrado.")

(-23.610278, -46.4672923)


In [61]:
df2 = df_resultado[df_resultado['LATITUDE'].isna()]

In [64]:
df2 = df2.drop(columns=['LATITUDE', 'LONGITUDE', 'ENDERECO_COMPLETO'])

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
import time
import requests
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

def obter_endereco_string_unica(cep):
    """
    Obtém endereço completo a partir do CEP usando a API ViaCEP
    """
    # Remove caracteres não numéricos do CEP
    cep = ''.join(filter(str.isdigit, cep))
    if len(cep) != 8:
        return "Erro: CEP inválido. Deve conter 8 dígitos."
    
    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Lança um erro para status de erro HTTP
        data = response.json()
        
        if "erro" in data and data["erro"] == True:
            return "Erro: CEP não encontrado."
        
        # Monta a string única do endereço
        logradouro = data.get('logradouro', 'Não informado')
        bairro = data.get('bairro', 'Não informado')
        localidade = data.get('localidade', 'Não informada')
        uf = data.get('uf', 'NI')
        
        # Exemplo de formatação: "Rua Exemplo, Bairro X, Cidade Y - Estado Z"
        endereco_completo = f"{logradouro}, {localidade} - {uf}"
        return endereco_completo
        
    except requests.exceptions.RequestException as e:
        return f"Erro na requisição: {e}"

def adicionar_endereco_completo(df):
    """
    Adiciona coluna ENDERECO_COMPLETO baseada na coluna ZIPCODE
    """
    print("Criando coluna ENDERECO_COMPLETO a partir dos CEPs...")
    enderecos_completos = []
    
    for index, zipcode in df['ZIP CODE'].items():
        print(f"Processando CEP linha {index}: {zipcode}")
        endereco = obter_endereco_string_unica(str(zipcode))
        enderecos_completos.append(endereco)
        print(f"  → {endereco}")
        
        # Pausa pequena para não sobrecarregar a API ViaCEP
        time.sleep(0.5)
    
    df['ENDERECO_COMPLETO'] = enderecos_completos
    return df

def geocode_addresses(df):
    """
    Adiciona colunas de latitude e longitude ao dataframe baseado na coluna ENDERECO_COMPLETO
    """
    # Inicializa o geocodificador com timeout personalizado
    geolocator = Nominatim(
        user_agent="geolocalizacao_app_v1.0",
        timeout=10  # Timeout de 10 segundos
    )
    
    # Cria listas para armazenar os resultados
    latitudes = []
    longitudes = []
    
    print("\nIniciando geocodificação dos endereços completos...")
    
    # Percorre cada endereço na coluna ENDERECO_COMPLETO
    for index, endereco_completo in df['ENDERECO_COMPLETO'].items():
        # Pula endereços com erro
        if str(endereco_completo).startswith("Erro:"):
            print(f"Pulando linha {index} (erro no CEP): {endereco_completo}")
            latitudes.append(None)
            longitudes.append(None)
            continue
            
        max_retries = 3
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            try:
                print(f"Geocodificando linha {index}: {endereco_completo} (tentativa {retry_count + 1})")
                
                # Geocodifica o endereço
                location = geolocator.geocode(endereco_completo, timeout=10)
                
                if location:
                    # Se encontrou localização, adiciona latitude e longitude
                    latitudes.append(location.latitude)
                    longitudes.append(location.longitude)
                    print(f"  ✓ Encontrado: {location.latitude}, {location.longitude}")
                    success = True
                else:
                    # Se não encontrou, adiciona valores nulos
                    latitudes.append(None)
                    longitudes.append(None)
                    print(f"  ✗ Endereço não encontrado")
                    success = True
                    
            except (GeocoderTimedOut, GeocoderServiceError) as e:
                retry_count += 1
                print(f"  ⚠ Erro de conexão (tentativa {retry_count}): {e}")
                
                if retry_count < max_retries:
                    wait_time = 2 ** retry_count  # Backoff exponencial
                    print(f"  ⏳ Aguardando {wait_time} segundos antes de tentar novamente...")
                    time.sleep(wait_time)
                else:
                    # Todas as tentativas falharam
                    latitudes.append(None)
                    longitudes.append(None)
                    print(f"  ✗ Falha após {max_retries} tentativas")
                    
            except Exception as e:
                # Outros erros não relacionados à conexão
                latitudes.append(None)
                longitudes.append(None)
                print(f"  ✗ Erro: {e}")
                success = True
        
        # Pausa para respeitar os limites da API (aumentada para maior estabilidade)
        time.sleep(2)
    
    # Adiciona as novas colunas ao dataframe
    df['LATITUDE'] = latitudes
    df['LONGITUDE'] = longitudes
    
    return df

def processar_geocodificacao_completa(df):
    """
    Função principal que executa todo o processo:
    1. Cria coluna ENDERECO_COMPLETO a partir de ZIPCODE
    2. Geocodifica usando ENDERECO_COMPLETO
    """
    print("=== INICIANDO PROCESSO DE GEOCODIFICAÇÃO COMPLETA ===\n")
    
    # Passo 1: Criar endereços completos a partir dos CEPs
    df_com_endereco = adicionar_endereco_completo(df.copy())
    
    # Mostrar estatísticas dos endereços obtidos
    enderecos_validos = df_com_endereco['ENDERECO_COMPLETO'].apply(
        lambda x: not str(x).startswith("Erro:")
    ).sum()
    total_enderecos = len(df_com_endereco)
    
    print(f"\n📊 Estatísticas dos CEPs:")
    print(f"   Total de registros: {total_enderecos}")
    print(f"   Endereços válidos obtidos: {enderecos_validos}")
    print(f"   Taxa de sucesso CEPs: {enderecos_validos/total_enderecos*100:.1f}%")
    
    # Passo 2: Geocodificar usando os endereços completos
    df_final = geocode_addresses(df_com_endereco)
    
    # Mostrar estatísticas finais
    coordenadas_validas = df_final['LATITUDE'].notna().sum()
    print(f"\n📊 Estatísticas finais:")
    print(f"   Coordenadas obtidas: {coordenadas_validas}")
    print(f"   Taxa de sucesso geocodificação: {coordenadas_validas/total_enderecos*100:.1f}%")
    
    return df_final

# Exemplo de uso:
df_resultado = processar_geocodificacao_completa(df)

# Para salvar o resultado:
df_resultado.to_csv('enderecos_geocodificados.csv', index=False)

# Para verificar os resultados:
# print(df_resultado[['ZIPCODE', 'ENDERECO_COMPLETO', 'LATITUDE', 'LONGITUDE']].head(10))

In [65]:
df_resultado2 = processar_geocodificacao_completa(df2)

=== INICIANDO PROCESSO DE GEOCODIFICAÇÃO COMPLETA ===

Criando coluna ENDERECO_COMPLETO a partir dos CEPs...
Processando CEP linha 0: 03348-000
  → Rua Capitão Ferraiuolo, São Paulo - SP
Processando CEP linha 1: 04336-160
  → Rua Doutor Alcides de Campos, São Paulo - SP
Processando CEP linha 2: 03018-010
  → Rua Coronel Emídio Piedade, São Paulo - SP
Processando CEP linha 3: 03018-000
  → Rua João Boemer, São Paulo - SP
Processando CEP linha 8: 03054-020
  → Rua Coronel Albino Bairão, São Paulo - SP
Processando CEP linha 9: 08265-150
  → Rua Iososuke Okaue, São Paulo - SP
Processando CEP linha 10: 03377-000
  → Avenida Renata, São Paulo - SP
Processando CEP linha 12: 03021-060
  → Rua Manuel Ramos Paiva, São Paulo - SP
Processando CEP linha 15: 03024-000
  → Rua Cachoeira, São Paulo - SP
Processando CEP linha 19: 03023-000
  → Rua Rio Bonito, São Paulo - SP
Processando CEP linha 26: 05338-050
  → Rua Henning Boilesen, São Paulo - SP
Processando CEP linha 28: 03023-000
  → Rua Rio Bonit

In [66]:
df_resultado2.to_excel('enderecos_geocodificados2.xlsx')


In [ ]:
Rua San José, Cotia - SP
Rua José Herculano de Oliveira Rosa, São Paulo - SP
Rua Framboesa de Heliópolis, São Paulo - SP
Via de Acesso Norte km 38, Cajamar - SP
Rua Manuel Aguilar, São Paulo - SP
Rua Quinta de Santa Luzia, São Paulo - SP
Rua Doutor Amaral Osório, São Paulo - SP
Rua Quinta de Santa Luzia, São Paulo - SP
Rua Quinta de Santa Luzia, São Paulo - SP
Rua Eudoro Lincoln Berlinck, São Paulo - SP

In [68]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geolocalização")
location = geolocator.geocode("Rua San José")

if location:  # Verifica se o endereço foi encontrado
    print((location.latitude, location.longitude))
else:
    print("Endereço não encontrado.")

(-20.8190157, -49.4513901)


In [79]:
def dms_to_decimal(graus, minutos, segundos, direcao):
    decimal = graus + (minutos / 60) + (segundos / 3600)
    if direcao in ["S", "W"]:
        decimal *= -1
    return decimal

# Exemplo para 23°36'53"S 46°53'00"W:
latitude = dms_to_decimal(23, 26, 42, "S")
longitude = dms_to_decimal(46, 34, 40, "W")

print(f"Latitude: {latitude}, Longitude: {longitude}")



Latitude: -23.445, Longitude: -46.57777777777778


In [ ]:
23°26'42"S 46°34'40"W

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from geopy.distance import geodesic
import folium
from folium import plugins
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

class LogisticsOptimizer:
    def __init__(self):
        self.df = None
        self.clusters = None
        self.simulation_results = None
        
    def generate_sample_data(self, n_points=150):
        """Gera dados simulados realistas para demonstração"""
        np.random.seed(42)
        
        # Coordenadas aproximadas de São Paulo e região metropolitana
        center_lat, center_lon = -23.5505, -46.6333
        
        # Criar 3-4 concentrações regionais
        concentrations = [
            (-23.5505, -46.6333, 40),  # Centro SP
            (-23.6821, -46.8755, 30),  # Osasco
            (-23.4538, -46.5333, 25),  # Guarulhos
            (-23.7937, -46.6816, 35),  # São Bernardo
            (-23.3200, -46.7311, 20),  # Jundiaí
        ]
        
        data = []
        seller_id = 1000
        
        for i, (lat_center, lon_center, n_stores) in enumerate(concentrations):
            for _ in range(n_stores):
                # Variação aleatória ao redor do centro
                lat = lat_center + np.random.normal(0, 0.05)  # ~5km de variação
                lon = lon_center + np.random.normal(0, 0.05)
                
                # Volume de pacotes com distribuição realista
                volume = max(1, int(np.random.lognormal(3, 1)))  # Média ~20, max ~200
                
                data.append({
                    'EXPECTPICKUP': pd.Timestamp.now().date(),
                    'CLIENTEseller_id': f'SELLER_{seller_id}',
                    'ADDRESS': f'Rua Exemplo {seller_id % 1000}',
                    'CITY': 'São Paulo' if i < 2 else ['Osasco', 'Guarulhos', 'São Bernardo', 'Jundiaí'][i-2],
                    'STATE': 'SP',
                    'ZIP_CODE': f'{8000 + i}{seller_id % 1000:03d}',
                    'AREA': f'Região {i+1}',
                    'ROTA': f'ROTA_{i+1}',
                    'VOLUME_TIKTOK_PICKUP': volume,
                    'ENDERECO_COMPLETO': f'Rua Exemplo {seller_id % 1000}, São Paulo, SP',
                    'LATITUDE': lat,
                    'LONGITUDE': lon
                })
                seller_id += 1
        
        self.df = pd.DataFrame(data)
        return self.df
    
    def load_data(self, df=None, filepath=None):
        """Carrega dados do usuário"""
        if df is not None:
            self.df = df.copy()
        elif filepath:
            self.df = pd.read_csv(filepath)
        else:
            self.df = self.generate_sample_data()
        
        # Limpeza básica
        self.df = self.df.dropna(subset=['LATITUDE', 'LONGITUDE', 'VOLUME_TIKTOK_PICKUP'])
        self.df = self.df[self.df['VOLUME_TIKTOK_PICKUP'] > 0]
        
        print(f"Dataset carregado: {len(self.df)} pontos de coleta")
        return self.df
    
    def exploratory_analysis(self):
        """Análise exploratória dos dados"""
        print("=== ANÁLISE EXPLORATÓRIA ===")
        print(f"Total de pontos: {len(self.df)}")
        print(f"Volume total de pacotes: {self.df['VOLUME_TIKTOK_PICKUP'].sum()}")
        print(f"Volume médio por ponto: {self.df['VOLUME_TIKTOK_PICKUP'].mean():.1f}")
        print(f"Volume mediano: {self.df['VOLUME_TIKTOK_PICKUP'].median():.1f}")
        print(f"Desvio padrão do volume: {self.df['VOLUME_TIKTOK_PICKUP'].std():.1f}")
        
        # Estatísticas geográficas
        lat_range = self.df['LATITUDE'].max() - self.df['LATITUDE'].min()
        lon_range = self.df['LONGITUDE'].max() - self.df['LONGITUDE'].min()
        print(f"Dispersão geográfica - Lat: {lat_range:.4f}°, Lon: {lon_range:.4f}°")
        
        return {
            'total_points': len(self.df),
            'total_volume': self.df['VOLUME_TIKTOK_PICKUP'].sum(),
            'avg_volume': self.df['VOLUME_TIKTOK_PICKUP'].mean(),
            'geographic_spread': (lat_range, lon_range)
        }
    
    def calculate_distance_matrix(self, sample_size=None):
        """Calcula matriz de distâncias geográficas"""
        df_sample = self.df.sample(n=min(sample_size or len(self.df), 200))  # Limita para performance
        
        n = len(df_sample)
        distance_matrix = np.zeros((n, n))
        
        coords = df_sample[['LATITUDE', 'LONGITUDE']].values
        
        for i in range(n):
            for j in range(i+1, n):
                dist = geodesic(coords[i], coords[j]).kilometers
                distance_matrix[i][j] = dist
                distance_matrix[j][i] = dist
        
        return distance_matrix, df_sample
    
    def find_optimal_clusters(self, max_clusters=12):
        """Encontra número ótimo de clusters usando método do cotovelo e silhueta"""
        X = self.df[['LATITUDE', 'LONGITUDE']].values
        weights = self.df['VOLUME_TIKTOK_PICKUP'].values
        
        # Normalização ponderada
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Adicionar peso como terceira dimensão (normalizado)
        weights_scaled = (weights - weights.min()) / (weights.max() - weights.min())
        X_weighted = np.column_stack([X_scaled, weights_scaled * 0.5])  # Peso reduzido para não dominar
        
        inertias = []
        silhouette_scores = []
        K_range = range(2, max_clusters + 1)
        
        for k in K_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            cluster_labels = kmeans.fit_predict(X_weighted)
            
            inertias.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X_weighted, cluster_labels))
        
        # Encontrar cotovelo
        deltas = np.diff(inertias)
        second_deltas = np.diff(deltas)
        elbow_idx = np.argmax(second_deltas) + 2  # +2 devido aos diffs
        optimal_k_elbow = K_range[elbow_idx] if elbow_idx < len(K_range) else K_range[-1]
        
        # Melhor silhueta
        optimal_k_silhouette = K_range[np.argmax(silhouette_scores)]
        
        # Compromisso entre os dois métodos
        optimal_k = int(np.mean([optimal_k_elbow, optimal_k_silhouette]))
        
        print(f"Número ótimo de clusters:")
        print(f"  - Método do cotovelo: {optimal_k_elbow}")
        print(f"  - Melhor silhueta: {optimal_k_silhouette}")
        print(f"  - Escolhido (média): {optimal_k}")
        
        return optimal_k, (K_range, inertias, silhouette_scores)
    
    def perform_clustering(self, n_clusters=None):
        """Executa clustering geográfico ponderado"""
        if n_clusters is None:
            n_clusters, _ = self.find_optimal_clusters()
        
        X = self.df[['LATITUDE', 'LONGITUDE']].values
        weights = self.df['VOLUME_TIKTOK_PICKUP'].values
        
        # Preparação dos dados
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        weights_scaled = (weights - weights.min()) / (weights.max() - weights.min())
        X_weighted = np.column_stack([X_scaled, weights_scaled * 0.3])
        
        # K-means principal
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
        cluster_labels = kmeans.fit_predict(X_weighted)
        
        # Adicionar clusters ao dataframe
        self.df['CLUSTER'] = cluster_labels
        
        # Calcular estatísticas por cluster
        cluster_stats = []
        for cluster_id in range(n_clusters):
            cluster_data = self.df[self.df['CLUSTER'] == cluster_id]
            
            # Centróide geográfico
            centroid_lat = cluster_data['LATITUDE'].mean()
            centroid_lon = cluster_data['LONGITUDE'].mean()
            
            # Raio máximo do cluster
            max_distance = 0
            for _, row in cluster_data.iterrows():
                dist = geodesic((centroid_lat, centroid_lon), 
                              (row['LATITUDE'], row['LONGITUDE'])).kilometers
                max_distance = max(max_distance, dist)
            
            stats = {
                'cluster_id': cluster_id,
                'n_points': len(cluster_data),
                'total_volume': cluster_data['VOLUME_TIKTOK_PICKUP'].sum(),
                'avg_volume_per_point': cluster_data['VOLUME_TIKTOK_PICKUP'].mean(),
                'centroid_lat': centroid_lat,
                'centroid_lon': centroid_lon,
                'max_radius_km': max_distance,
                'cities': cluster_data['CITY'].value_counts().to_dict()
            }
            cluster_stats.append(stats)
        
        self.clusters = pd.DataFrame(cluster_stats)
        
        print("\n=== RESULTADOS DO CLUSTERING ===")
        for _, cluster in self.clusters.iterrows():
            print(f"Cluster {cluster['cluster_id']}:")
            print(f"  - Pontos: {cluster['n_points']}")
            print(f"  - Volume total: {cluster['total_volume']}")
            print(f"  - Raio máximo: {cluster['max_radius_km']:.1f} km")
            print(f"  - Principais cidades: {list(cluster['cities'].keys())[:3]}")
            print()
        
        return self.clusters
    
    def monte_carlo_simulation(self, n_simulations=10000):
        """Simulação Monte Carlo para dimensionamento de frota"""
        if self.clusters is None:
            raise ValueError("Execute o clustering primeiro!")
        
        # Parâmetros da simulação
        vehicle_capacities = [100, 150, 300]
        trips_per_day = [1, 2]
        
        # Parâmetros de tempo (em horas)
        base_pickup_time = 0.25  # 15 min por ponto
        travel_time_per_km = 0.05  # 3 min por km
        working_hours_per_day = 8
        
        simulation_results = []
        
        for _, cluster in self.clusters.iterrows():
            cluster_id = cluster['cluster_id']
            total_volume = cluster['total_volume']
            n_points = cluster['n_points']
            max_radius = cluster['max_radius_km']
            
            print(f"Simulando Cluster {cluster_id}...")
            
            for capacity in vehicle_capacities:
                for trips in trips_per_day:
                    daily_capacity = capacity * trips
                    vehicles_needed = []
                    
                    for sim in range(n_simulations):
                        # Variações aleatórias
                        volume_variation = np.random.uniform(0.85, 1.15)  # ±15%
                        time_variation = np.random.uniform(0.8, 1.2)     # ±20%
                        traffic_factor = np.random.uniform(1.0, 1.5)     # Trânsito
                        
                        adjusted_volume = int(total_volume * volume_variation)
                        
                        # Tempo estimado por veículo
                        pickup_time = n_points * base_pickup_time * time_variation
                        travel_time = max_radius * 2 * travel_time_per_km * traffic_factor
                        total_time_per_trip = pickup_time + travel_time
                        
                        # Verificar se cabe no dia de trabalho
                        if total_time_per_trip * trips > working_hours_per_day:
                            # Reduzir eficiência se não couber no tempo
                            efficiency_penalty = (total_time_per_trip * trips) / working_hours_per_day
                            daily_capacity = int(daily_capacity / efficiency_penalty)
                        
                        # Calcular veículos necessários
                        vehicles = max(1, int(np.ceil(adjusted_volume / daily_capacity)))
                        vehicles_needed.append(vehicles)
                    
                    # Estatísticas da simulação
                    result = {
                        'cluster_id': cluster_id,
                        'capacity': capacity,
                        'trips': trips,
                        'daily_capacity': capacity * trips,
                        'min_vehicles': np.percentile(vehicles_needed, 5),
                        'median_vehicles': np.percentile(vehicles_needed, 50),
                        'max_vehicles': np.percentile(vehicles_needed, 95),
                        'mean_vehicles': np.mean(vehicles_needed),
                        'std_vehicles': np.std(vehicles_needed)
                    }
                    simulation_results.append(result)
        
        self.simulation_results = pd.DataFrame(simulation_results)
        return self.simulation_results
    
    def generate_fleet_recommendations(self):
        """Gera recomendações finais de frota"""
        if self.simulation_results is None:
            raise ValueError("Execute a simulação Monte Carlo primeiro!")
        
        recommendations = []
        
        for cluster_id in self.clusters['cluster_id'].unique():
            cluster_sims = self.simulation_results[
                self.simulation_results['cluster_id'] == cluster_id
            ]
            
            # Cenários
            conservative = cluster_sims[
                (cluster_sims['capacity'] == 100) & (cluster_sims['trips'] == 1)
            ].iloc[0]
            
            moderate = cluster_sims[
                (cluster_sims['capacity'] == 150) & (cluster_sims['trips'] == 1)
            ].iloc[0]
            
            optimistic = cluster_sims[
                (cluster_sims['capacity'] == 300) & (cluster_sims['trips'] == 2)
            ].iloc[0]
            
            cluster_info = self.clusters[self.clusters['cluster_id'] == cluster_id].iloc[0]
            
            recommendation = {
                'cluster_id': cluster_id,
                'total_volume': cluster_info['total_volume'],
                'n_points': cluster_info['n_points'],
                'conservative_range': f"{int(conservative['min_vehicles'])}-{int(conservative['max_vehicles'])}",
                'moderate_range': f"{int(moderate['min_vehicles'])}-{int(moderate['max_vehicles'])}",
                'optimistic_range': f"{int(optimistic['min_vehicles'])}-{int(optimistic['max_vehicles'])}",
                'recommended_scenario': 'moderate',
                'recommended_vehicles': f"{int(moderate['min_vehicles'])}-{int(moderate['max_vehicles'])}",
                'main_cities': ', '.join(list(cluster_info['cities'].keys())[:3])
            }
            recommendations.append(recommendation)
        
        recommendations_df = pd.DataFrame(recommendations)
        
        print("\n=== RECOMENDAÇÕES DE FROTA ===")
        print(recommendations_df.to_string(index=False))
        
        return recommendations_df
    
    def create_interactive_map(self):
        """Cria mapa interativo com clusters"""
        if self.df is None or 'CLUSTER' not in self.df.columns:
            raise ValueError("Execute o clustering primeiro!")
        
        # Centro do mapa
        center_lat = self.df['LATITUDE'].mean()
        center_lon = self.df['LONGITUDE'].mean()
        
        # Criar mapa
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=10,
            tiles='OpenStreetMap'
        )
        
        # Cores para clusters
        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
                 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 
                 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen']
        
        # Adicionar pontos por cluster
        for cluster_id in sorted(self.df['CLUSTER'].unique()):
            cluster_data = self.df[self.df['CLUSTER'] == cluster_id]
            color = colors[cluster_id % len(colors)]
            
            for _, row in cluster_data.iterrows():
                folium.CircleMarker(
                    location=[row['LATITUDE'], row['LONGITUDE']],
                    radius=max(3, min(15, row['VOLUME_TIKTOK_PICKUP'] / 10)),
                    popup=f"""
                    <b>Cluster {cluster_id}</b><br>
                    Seller: {row['CLIENTEseller_id']}<br>
                    Volume: {row['VOLUME_TIKTOK_PICKUP']} pacotes<br>
                    Cidade: {row['CITY']}<br>
                    Endereço: {row['ADDRESS']}
                    """,
                    color=color,
                    fill=True,
                    fillOpacity=0.6
                ).add_to(m)
        
        # Adicionar centróides dos clusters
        for _, cluster in self.clusters.iterrows():
            folium.Marker(
                location=[cluster['centroid_lat'], cluster['centroid_lon']],
                popup=f"""
                <b>Centro do Cluster {cluster['cluster_id']}</b><br>
                Pontos: {cluster['n_points']}<br>
                Volume Total: {cluster['total_volume']}<br>
                Raio: {cluster['max_radius_km']:.1f} km
                """,
                icon=folium.Icon(color='black', icon='info-sign')
            ).add_to(m)
            
            # Círculo de cobertura
            folium.Circle(
                location=[cluster['centroid_lat'], cluster['centroid_lon']],
                radius=cluster['max_radius_km'] * 1000,  # metros
                color=colors[cluster['cluster_id'] % len(colors)],
                fill=False,
                opacity=0.3
            ).add_to(m)
        
        return m
    
    def run_complete_analysis(self, df=None, n_clusters=None):
        """Executa análise completa"""
        print("🚚 INICIANDO ANÁLISE DE OTIMIZAÇÃO LOGÍSTICA")
        print("=" * 50)
        
        # 1. Carregar dados
        self.load_data(df)
        
        # 2. Análise exploratória
        self.exploratory_analysis()
        
        # 3. Clustering
        print("\n📍 EXECUTANDO CLUSTERING...")
        self.perform_clustering(n_clusters)
        
        # 4. Simulação Monte Carlo
        print("\n🎲 EXECUTANDO SIMULAÇÃO MONTE CARLO...")
        self.monte_carlo_simulation()
        
        # 5. Recomendações
        print("\n📊 GERANDO RECOMENDAÇÕES...")
        recommendations = self.generate_fleet_recommendations()
        
        # 6. Mapa
        print("\n🗺️  CRIANDO MAPA INTERATIVO...")
        map_viz = self.create_interactive_map()
        
        return {
            'data': self.df,
            'clusters': self.clusters,
            'simulation_results': self.simulation_results,
            'recommendations': recommendations,
            'map': map_viz
        }



In [ ]:
# Exemplo de uso
if __name__ == "__main__":
    # Inicializar analisador
    optimizer = LogisticsOptimizer()
    
    # Executar análise completa
    results = optimizer.run_complete_analysis()
    
    # Salvar mapa
    results['map'].save('mapa_clusters_logistica.html')
    print("\n✅ Análise concluída! Mapa salvo como 'mapa_clusters_logistica.html'")
    
    # Exibir resumo final
    print("\n" + "="*50)
    print("📋 RESUMO EXECUTIVO")
    print("="*50)
    total_vehicles_conservative = results['recommendations']['conservative_range'].str.split('-').str[1].astype(int).sum()
    total_vehicles_optimistic = results['recommendations']['optimistic_range'].str.split('-').str[0].astype(int).sum()
    
    print(f"🔢 Total de clusters identificados: {len(results['clusters'])}")
    print(f"📦 Volume total de pacotes: {results['data']['VOLUME_TIKTOK_PICKUP'].sum()}")
    print(f"🚛 Estimativa de veículos necessários: {total_vehicles_optimistic} - {total_vehicles_conservative}")
    print(f"💡 Cenário recomendado: Moderado (150 pacotes/veículo, 1 viagem/dia)")

In [88]:
# Opção B: Arquivo Excel
df = pd.read_excel('/Users/igorbione/Documents/Projeto_Logistica_TT/data/processed/Dados_logistica.xlsx')

# Verificar se carregou corretamente
print(f"Dataset carregado: {len(df)} linhas")
print(df.head())

Dataset carregado: 270 linhas
  EXPECT PICKUP CLIENTE     CLIENTEseller_id  \
0    2025-05-26  TIKTOK  7496179179138159616   
1    2025-05-26  TIKTOK  7496161546569480192   
2    2025-05-26  TIKTOK  7496147585319799808   
3    2025-05-26  TIKTOK  7496162496353829888   
4    2025-05-26  TIKTOK  7496159695603600384   

                                             ADDRESS       CITY      STATE  \
0                         RUA CAPITÃO FERRAIUOLO 367  São Paulo  São Paulo   
1                    RUA DOUTOR ALCIDES DE CAMPOS 35  São Paulo  São Paulo   
2  RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...  São Paulo  São Paulo   
3                               RUA JOÃO BOEMER 1259  São Paulo  São Paulo   
4                      RUA CORONEL ALBINO BAIRÃO 177  São Paulo  São Paulo   

    ZIP CODE            AREA                ROTA  VOLUME_TIKTOK_PICKUP  \
0  03348-000  Vila Invernada  SP Capital - Leste                     3   
1  04336-160  Americanópolis    SP Capital - Sul                

In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from geopy.distance import geodesic
import folium
from folium import plugins
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

class LogisticsOptimizer:
    def __init__(self):
        self.df = None
        self.clusters = None
        self.simulation_results = None
        
    def generate_sample_data(self, n_points=150):
        """Gera dados simulados realistas para demonstração"""
        np.random.seed(42)
        
        # Coordenadas aproximadas de São Paulo e região metropolitana
        center_lat, center_lon = -23.5505, -46.6333
        
        # Criar 3-4 concentrações regionais
        concentrations = [
            (-23.5505, -46.6333, 40),  # Centro SP
            (-23.6821, -46.8755, 30),  # Osasco
            (-23.4538, -46.5333, 25),  # Guarulhos
            (-23.7937, -46.6816, 35),  # São Bernardo
            (-23.3200, -46.7311, 20),  # Jundiaí
        ]
        
        data = []
        seller_id = 1000
        
        for i, (lat_center, lon_center, n_stores) in enumerate(concentrations):
            for _ in range(n_stores):
                # Variação aleatória ao redor do centro
                lat = lat_center + np.random.normal(0, 0.05)  # ~5km de variação
                lon = lon_center + np.random.normal(0, 0.05)
                
                # Volume de pacotes com distribuição realista
                volume = max(1, int(np.random.lognormal(3, 1)))  # Média ~20, max ~200
                
                data.append({
                    'EXPECTPICKUP': pd.Timestamp.now().date(),
                    'CLIENTEseller_id': f'SELLER_{seller_id}',
                    'ADDRESS': f'Rua Exemplo {seller_id % 1000}',
                    'CITY': 'São Paulo' if i < 2 else ['Osasco', 'Guarulhos', 'São Bernardo', 'Jundiaí'][i-2],
                    'STATE': 'SP',
                    'ZIP_CODE': f'{8000 + i}{seller_id % 1000:03d}',
                    'AREA': f'Região {i+1}',
                    'ROTA': f'ROTA_{i+1}',
                    'VOLUME_TIKTOK_PICKUP': volume,
                    'ENDERECO_COMPLETO': f'Rua Exemplo {seller_id % 1000}, São Paulo, SP',
                    'LATITUDE': lat,
                    'LONGITUDE': lon
                })
                seller_id += 1
        
        self.df = pd.DataFrame(data)
        return self.df
    
    def load_data(self, df=None, filepath=None):
        """Carrega dados do usuário"""
        if df is not None:
            self.df = df.copy()
        elif filepath:
            self.df = pd.read_csv(filepath)
        else:
            self.df = self.generate_sample_data()
        
        # Limpeza básica
        self.df = self.df.dropna(subset=['LATITUDE', 'LONGITUDE', 'VOLUME_TIKTOK_PICKUP'])
        self.df = self.df[self.df['VOLUME_TIKTOK_PICKUP'] > 0]
        
        # Verificações de segurança
        if len(self.df) == 0:
            raise ValueError("Nenhum dado válido encontrado após limpeza. Verifique as colunas LATITUDE, LONGITUDE e VOLUME_TIKTOK_PICKUP")
        
        # Verificar se as coordenadas são válidas
        invalid_coords = (
            (self.df['LATITUDE'].abs() > 90) | 
            (self.df['LONGITUDE'].abs() > 180) |
            (self.df['LATITUDE'].isna()) |
            (self.df['LONGITUDE'].isna())
        )
        
        if invalid_coords.sum() > 0:
            print(f"⚠️  Removendo {invalid_coords.sum()} pontos com coordenadas inválidas")
            self.df = self.df[~invalid_coords]
        
        print(f"Dataset carregado: {len(self.df)} pontos de coleta")
        print(f"Coordenadas - Lat: {self.df['LATITUDE'].min():.4f} a {self.df['LATITUDE'].max():.4f}")
        print(f"Coordenadas - Lon: {self.df['LONGITUDE'].min():.4f} a {self.df['LONGITUDE'].max():.4f}")
        print(f"Volume - Min: {self.df['VOLUME_TIKTOK_PICKUP'].min()}, Max: {self.df['VOLUME_TIKTOK_PICKUP'].max()}")
        
        return self.df
    
    def exploratory_analysis(self):
        """Análise exploratória dos dados"""
        print("=== ANÁLISE EXPLORATÓRIA ===")
        print(f"Total de pontos: {len(self.df)}")
        print(f"Volume total de pacotes: {self.df['VOLUME_TIKTOK_PICKUP'].sum()}")
        print(f"Volume médio por ponto: {self.df['VOLUME_TIKTOK_PICKUP'].mean():.1f}")
        print(f"Volume mediano: {self.df['VOLUME_TIKTOK_PICKUP'].median():.1f}")
        print(f"Desvio padrão do volume: {self.df['VOLUME_TIKTOK_PICKUP'].std():.1f}")
        
        # Estatísticas geográficas
        lat_range = self.df['LATITUDE'].max() - self.df['LATITUDE'].min()
        lon_range = self.df['LONGITUDE'].max() - self.df['LONGITUDE'].min()
        print(f"Dispersão geográfica - Lat: {lat_range:.4f}°, Lon: {lon_range:.4f}°")
        
        return {
            'total_points': len(self.df),
            'total_volume': self.df['VOLUME_TIKTOK_PICKUP'].sum(),
            'avg_volume': self.df['VOLUME_TIKTOK_PICKUP'].mean(),
            'geographic_spread': (lat_range, lon_range)
        }
    
    def calculate_distance_matrix(self, sample_size=None):
        """Calcula matriz de distâncias geográficas"""
        df_sample = self.df.sample(n=min(sample_size or len(self.df), 200))  # Limita para performance
        
        n = len(df_sample)
        distance_matrix = np.zeros((n, n))
        
        coords = df_sample[['LATITUDE', 'LONGITUDE']].values
        
        for i in range(n):
            for j in range(i+1, n):
                dist = geodesic(coords[i], coords[j]).kilometers
                distance_matrix[i][j] = dist
                distance_matrix[j][i] = dist
        
        return distance_matrix, df_sample
    
    def find_optimal_clusters(self, max_clusters=12):
        """Encontra número ótimo de clusters usando método do cotovelo e silhueta"""
        X = self.df[['LATITUDE', 'LONGITUDE']].values
        weights = self.df['VOLUME_TIKTOK_PICKUP'].values
        
        # Normalização ponderada
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Adicionar peso como terceira dimensão (normalizado)
        weights_scaled = (weights - weights.min()) / (weights.max() - weights.min())
        X_weighted = np.column_stack([X_scaled, weights_scaled * 0.5])  # Peso reduzido para não dominar
        
        inertias = []
        silhouette_scores = []
        K_range = range(2, max_clusters + 1)
        
        for k in K_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            cluster_labels = kmeans.fit_predict(X_weighted)
            
            inertias.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X_weighted, cluster_labels))
        
        # Encontrar cotovelo
        deltas = np.diff(inertias)
        second_deltas = np.diff(deltas)
        elbow_idx = np.argmax(second_deltas) + 2  # +2 devido aos diffs
        optimal_k_elbow = K_range[elbow_idx] if elbow_idx < len(K_range) else K_range[-1]
        
        # Melhor silhueta
        optimal_k_silhouette = K_range[np.argmax(silhouette_scores)]
        
        # Compromisso entre os dois métodos
        optimal_k = int(np.mean([optimal_k_elbow, optimal_k_silhouette]))
        
        print(f"Número ótimo de clusters:")
        print(f"  - Método do cotovelo: {optimal_k_elbow}")
        print(f"  - Melhor silhueta: {optimal_k_silhouette}")
        print(f"  - Escolhido (média): {optimal_k}")
        
        return optimal_k, (K_range, inertias, silhouette_scores)
    
    def perform_clustering(self, n_clusters=None):
        """Executa clustering geográfico ponderado"""
        if n_clusters is None:
            n_clusters, _ = self.find_optimal_clusters()
        
        X = self.df[['LATITUDE', 'LONGITUDE']].values
        weights = self.df['VOLUME_TIKTOK_PICKUP'].values
        
        # Preparação dos dados
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        weights_scaled = (weights - weights.min()) / (weights.max() - weights.min())
        X_weighted = np.column_stack([X_scaled, weights_scaled * 0.3])
        
        # K-means principal
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
        cluster_labels = kmeans.fit_predict(X_weighted)
        
        # Adicionar clusters ao dataframe
        self.df['CLUSTER'] = cluster_labels
        
        # Calcular estatísticas por cluster
        cluster_stats = []
        for cluster_id in range(n_clusters):
            cluster_data = self.df[self.df['CLUSTER'] == cluster_id]
            
            # Centróide geográfico
            centroid_lat = cluster_data['LATITUDE'].mean()
            centroid_lon = cluster_data['LONGITUDE'].mean()
            
            # Raio máximo do cluster
            max_distance = 0
            for _, row in cluster_data.iterrows():
                dist = geodesic((centroid_lat, centroid_lon), 
                              (row['LATITUDE'], row['LONGITUDE'])).kilometers
                max_distance = max(max_distance, dist)
            
            stats = {
                'cluster_id': cluster_id,
                'n_points': len(cluster_data),
                'total_volume': cluster_data['VOLUME_TIKTOK_PICKUP'].sum(),
                'avg_volume_per_point': cluster_data['VOLUME_TIKTOK_PICKUP'].mean(),
                'centroid_lat': centroid_lat,
                'centroid_lon': centroid_lon,
                'max_radius_km': max_distance,
                'cities': cluster_data['CITY'].value_counts().to_dict()
            }
            cluster_stats.append(stats)
        
        self.clusters = pd.DataFrame(cluster_stats)
        
        print("\n=== RESULTADOS DO CLUSTERING ===")
        for _, cluster in self.clusters.iterrows():
            print(f"Cluster {cluster['cluster_id']}:")
            print(f"  - Pontos: {cluster['n_points']}")
            print(f"  - Volume total: {cluster['total_volume']}")
            print(f"  - Raio máximo: {cluster['max_radius_km']:.1f} km")
            print(f"  - Principais cidades: {list(cluster['cities'].keys())[:3]}")
            print()
        
        return self.clusters
    
    def monte_carlo_simulation(self, n_simulations=5000):
        """Simulação Monte Carlo para dimensionamento de frota"""
        if self.clusters is None:
            raise ValueError("Execute o clustering primeiro!")
        
        # Parâmetros da simulação
        vehicle_capacities = [100, 150, 300]
        trips_per_day = [1, 2]
        
        # Parâmetros de tempo (em horas) - mais conservadores
        base_pickup_time = 0.25  # 15 min por ponto
        travel_time_per_km = 0.05  # 3 min por km
        working_hours_per_day = 8
        max_reasonable_time = working_hours_per_day * 0.8  # 80% do tempo máximo
        
        simulation_results = []
        
        for _, cluster in self.clusters.iterrows():
            cluster_id = cluster['cluster_id']
            total_volume = cluster['total_volume']
            n_points = cluster['n_points']
            max_radius = cluster['max_radius_km']
            
            print(f"Simulando Cluster {cluster_id} (Volume: {total_volume}, Pontos: {n_points})...")
            
            for capacity in vehicle_capacities:
                for trips in trips_per_day:
                    daily_capacity = capacity * trips
                    vehicles_needed = []
                    
                    for sim in range(n_simulations):
                        # Variações aleatórias mais conservadoras
                        volume_variation = np.random.uniform(0.9, 1.1)    # ±10%
                        time_variation = np.random.uniform(0.9, 1.1)     # ±10%
                        traffic_factor = np.random.uniform(1.0, 1.3)     # Trânsito mais moderado
                        
                        adjusted_volume = max(1, int(total_volume * volume_variation))
                        
                        # Tempo estimado por veículo
                        pickup_time = n_points * base_pickup_time * time_variation
                        travel_time = max_radius * 2 * travel_time_per_km * traffic_factor
                        total_time_per_trip = pickup_time + travel_time
                        
                        # Capacidade diária inicial
                        current_daily_capacity = daily_capacity
                        
                        # Verificar se cabe no dia de trabalho
                        total_daily_time = total_time_per_trip * trips
                        if total_daily_time > max_reasonable_time:
                            # Reduzir eficiência se não couber no tempo
                            efficiency_penalty = total_daily_time / max_reasonable_time
                            current_daily_capacity = max(10, int(current_daily_capacity / efficiency_penalty))
                        
                        # Garantir que daily_capacity nunca seja zero ou muito baixo
                        current_daily_capacity = max(5, current_daily_capacity)
                        
                        # Calcular veículos necessários
                        vehicles = max(1, int(np.ceil(adjusted_volume / current_daily_capacity)))
                        
                        # Limite máximo razoável de veículos por cluster
                        vehicles = min(vehicles, max(10, n_points))
                        
                        vehicles_needed.append(vehicles)
                    
                    # Estatísticas da simulação
                    result = {
                        'cluster_id': cluster_id,
                        'capacity': capacity,
                        'trips': trips,
                        'daily_capacity': capacity * trips,
                        'min_vehicles': int(np.percentile(vehicles_needed, 5)),
                        'median_vehicles': int(np.percentile(vehicles_needed, 50)),
                        'max_vehicles': int(np.percentile(vehicles_needed, 95)),
                        'mean_vehicles': np.mean(vehicles_needed),
                        'std_vehicles': np.std(vehicles_needed)
                    }
                    simulation_results.append(result)
        
        self.simulation_results = pd.DataFrame(simulation_results)
        print(f"✅ Simulação concluída para {len(self.clusters)} clusters")
        return self.simulation_results
    
    def generate_fleet_recommendations(self):
        """Gera recomendações finais de frota"""
        if self.simulation_results is None:
            raise ValueError("Execute a simulação Monte Carlo primeiro!")
        
        recommendations = []
        
        for cluster_id in self.clusters['cluster_id'].unique():
            cluster_sims = self.simulation_results[
                self.simulation_results['cluster_id'] == cluster_id
            ]
            
            # Cenários
            conservative = cluster_sims[
                (cluster_sims['capacity'] == 100) & (cluster_sims['trips'] == 1)
            ].iloc[0]
            
            moderate = cluster_sims[
                (cluster_sims['capacity'] == 150) & (cluster_sims['trips'] == 1)
            ].iloc[0]
            
            optimistic = cluster_sims[
                (cluster_sims['capacity'] == 300) & (cluster_sims['trips'] == 2)
            ].iloc[0]
            
            cluster_info = self.clusters[self.clusters['cluster_id'] == cluster_id].iloc[0]
            
            recommendation = {
                'cluster_id': cluster_id,
                'total_volume': cluster_info['total_volume'],
                'n_points': cluster_info['n_points'],
                'conservative_range': f"{int(conservative['min_vehicles'])}-{int(conservative['max_vehicles'])}",
                'moderate_range': f"{int(moderate['min_vehicles'])}-{int(moderate['max_vehicles'])}",
                'optimistic_range': f"{int(optimistic['min_vehicles'])}-{int(optimistic['max_vehicles'])}",
                'recommended_scenario': 'moderate',
                'recommended_vehicles': f"{int(moderate['min_vehicles'])}-{int(moderate['max_vehicles'])}",
                'main_cities': ', '.join(list(cluster_info['cities'].keys())[:3])
            }
            recommendations.append(recommendation)
        
        recommendations_df = pd.DataFrame(recommendations)
        
        print("\n=== RECOMENDAÇÕES DE FROTA ===")
        print(recommendations_df.to_string(index=False))
        
        return recommendations_df
    
    def create_interactive_map(self):
        """Cria mapa interativo com clusters"""
        if self.df is None or 'CLUSTER' not in self.df.columns:
            raise ValueError("Execute o clustering primeiro!")
        
        # Centro do mapa
        center_lat = self.df['LATITUDE'].mean()
        center_lon = self.df['LONGITUDE'].mean()
        
        # Criar mapa
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=10,
            tiles='OpenStreetMap'
        )
        
        # Cores para clusters
        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
                 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 
                 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen']
        
        # Adicionar pontos por cluster
        for cluster_id in sorted(self.df['CLUSTER'].unique()):
            cluster_data = self.df[self.df['CLUSTER'] == cluster_id]
            color = colors[cluster_id % len(colors)]
            
            for _, row in cluster_data.iterrows():
                folium.CircleMarker(
                    location=[row['LATITUDE'], row['LONGITUDE']],
                    radius=max(3, min(15, row['VOLUME_TIKTOK_PICKUP'] / 10)),
                    popup=f"""
                    <b>Cluster {cluster_id}</b><br>
                    Seller: {row['CLIENTEseller_id']}<br>
                    Volume: {row['VOLUME_TIKTOK_PICKUP']} pacotes<br>
                    Cidade: {row['CITY']}<br>
                    Endereço: {row['ADDRESS']}
                    """,
                    color=color,
                    fill=True,
                    fillOpacity=0.6
                ).add_to(m)
        
        # Adicionar centróides dos clusters
        for _, cluster in self.clusters.iterrows():
            folium.Marker(
                location=[cluster['centroid_lat'], cluster['centroid_lon']],
                popup=f"""
                <b>Centro do Cluster {cluster['cluster_id']}</b><br>
                Pontos: {cluster['n_points']}<br>
                Volume Total: {cluster['total_volume']}<br>
                Raio: {cluster['max_radius_km']:.1f} km
                """,
                icon=folium.Icon(color='black', icon='info-sign')
            ).add_to(m)
            
            # Círculo de cobertura
            folium.Circle(
                location=[cluster['centroid_lat'], cluster['centroid_lon']],
                radius=cluster['max_radius_km'] * 1000,  # metros
                color=colors[cluster['cluster_id'] % len(colors)],
                fill=False,
                opacity=0.3
            ).add_to(m)
        
        return m
    
    def run_complete_analysis(self, df=None, n_clusters=None):
        """Executa análise completa"""
        print("🚚 INICIANDO ANÁLISE DE OTIMIZAÇÃO LOGÍSTICA")
        print("=" * 50)
        
        # 1. Carregar dados
        self.load_data(df)
        
        # 2. Análise exploratória
        self.exploratory_analysis()
        
        # 3. Clustering
        print("\n📍 EXECUTANDO CLUSTERING...")
        self.perform_clustering(n_clusters)
        
        # 4. Simulação Monte Carlo
        print("\n🎲 EXECUTANDO SIMULAÇÃO MONTE CARLO...")
        self.monte_carlo_simulation()
        
        # 5. Recomendações
        print("\n📊 GERANDO RECOMENDAÇÕES...")
        recommendations = self.generate_fleet_recommendations()
        
        # 6. Mapa
        print("\n🗺️  CRIANDO MAPA INTERATIVO...")
        map_viz = self.create_interactive_map()
        
        return {
            'data': self.df,
            'clusters': self.clusters,
            'simulation_results': self.simulation_results,
            'recommendations': recommendations,
            'map': map_viz
        }



In [89]:
#from logistics_optimizer import LogisticsOptimizer  # Se salvou em arquivo separado
# OU copie todo o código da classe LogisticsOptimizer

# Inicializar
optimizer = LogisticsOptimizer()

# Executar análise completa com seus dados
results = optimizer.run_complete_analysis(df=df)

# OU testar com dados simulados primeiro
results = optimizer.run_complete_analysis()  

🚚 INICIANDO ANÁLISE DE OTIMIZAÇÃO LOGÍSTICA
Dataset carregado: 270 pontos de coleta
Coordenadas - Lat: -23.8746 a -23.3400
Coordenadas - Lon: -46.9232 a -46.4111
Volume - Min: 1, Max: 796
=== ANÁLISE EXPLORATÓRIA ===
Total de pontos: 270
Volume total de pacotes: 5083
Volume médio por ponto: 18.8
Volume mediano: 3.0
Desvio padrão do volume: 70.7
Dispersão geográfica - Lat: 0.5346°, Lon: 0.5121°

📍 EXECUTANDO CLUSTERING...
Número ótimo de clusters:
  - Método do cotovelo: 4
  - Melhor silhueta: 2
  - Escolhido (média): 3

=== RESULTADOS DO CLUSTERING ===
Cluster 0:
  - Pontos: 45
  - Volume total: 243
  - Raio máximo: 12.2 km
  - Principais cidades: ['São Paulo', 'Santo André']

Cluster 1:
  - Pontos: 46
  - Volume total: 293
  - Raio máximo: 26.1 km
  - Principais cidades: ['São Paulo', 'Cotia', 'Jandira']

Cluster 2:
  - Pontos: 179
  - Volume total: 4547
  - Raio máximo: 30.4 km
  - Principais cidades: ['São Paulo', 'Cajamar']


🎲 EXECUTANDO SIMULAÇÃO MONTE CARLO...
Simulando Cluster 

In [90]:
mapa = optimizer.create_interactive_map()

In [91]:
mapa

In [92]:
optimizer = LogisticsOptimizer()
optimizer.load_data(df=df)
optimizer.exploratory_analysis()
clusters = optimizer.perform_clustering()
simulation = optimizer.monte_carlo_simulation()
recommendations = optimizer.generate_fleet_recommendations()
mapa = optimizer.create_interactive_map()

Dataset carregado: 270 pontos de coleta
Coordenadas - Lat: -23.8746 a -23.3400
Coordenadas - Lon: -46.9232 a -46.4111
Volume - Min: 1, Max: 796
=== ANÁLISE EXPLORATÓRIA ===
Total de pontos: 270
Volume total de pacotes: 5083
Volume médio por ponto: 18.8
Volume mediano: 3.0
Desvio padrão do volume: 70.7
Dispersão geográfica - Lat: 0.5346°, Lon: 0.5121°
Número ótimo de clusters:
  - Método do cotovelo: 4
  - Melhor silhueta: 2
  - Escolhido (média): 3

=== RESULTADOS DO CLUSTERING ===
Cluster 0:
  - Pontos: 45
  - Volume total: 243
  - Raio máximo: 12.2 km
  - Principais cidades: ['São Paulo', 'Santo André']

Cluster 1:
  - Pontos: 46
  - Volume total: 293
  - Raio máximo: 26.1 km
  - Principais cidades: ['São Paulo', 'Cotia', 'Jandira']

Cluster 2:
  - Pontos: 179
  - Volume total: 4547
  - Raio máximo: 30.4 km
  - Principais cidades: ['São Paulo', 'Cajamar']

Simulando Cluster 0 (Volume: 243, Pontos: 45)...
Simulando Cluster 1 (Volume: 293, Pontos: 46)...
Simulando Cluster 2 (Volume: 45

In [95]:
simulation

,cluster_id,capacity,trips,daily_capacity,min_vehicles,median_vehicles,max_vehicles,mean_vehicles,std_vehicles
0,0,100,1,100,5,5,6,5.3376,0.479611
1,0,100,2,200,5,5,6,5.3200,0.473286
2,0,150,1,150,3,4,4,3.7676,0.422363
3,0,150,2,300,3,4,4,3.7890,0.408018
4,0,300,1,300,2,2,2,2.0000,0.000000
5,0,300,2,600,2,2,2,2.0000,0.000000
6,1,100,1,100,6,7,8,7.1806,0.576180
7,1,100,2,200,6,7,8,7.1970,0.578784
8,1,150,1,150,4,5,6,4.9536,0.394268
9,1,150,2,300,4,5,6,4.9484,0.393113


In [96]:
optimizer.exploratory_analysis()

=== ANÁLISE EXPLORATÓRIA ===
Total de pontos: 270
Volume total de pacotes: 5083
Volume médio por ponto: 18.8
Volume mediano: 3.0
Desvio padrão do volume: 70.7
Dispersão geográfica - Lat: 0.5346°, Lon: 0.5121°


{'total_points': 270,
 'total_volume': 5083,
 'avg_volume': 18.825925925925926,
 'geographic_spread': (0.5345587999999992, 0.5121006000000037)}

In [98]:
def export_map_html(self, filename="mapa_clusters.html"):
    """Exporta o mapa como arquivo HTML"""
    m = self.create_interactive_map()
    m.save(filename)
    print(f"Mapa salvo como {filename}")
    return filename

mapa = optimizer.create_interactive_map()
mapa_file = mapa.export_map_html("mapa_cliente_2024.html")

AttributeError: 'Map' object has no attribute 'export_map_html'

In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from geopy.distance import geodesic
import folium
from folium import plugins
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime
import os
warnings.filterwarnings('ignore')

class LogisticsOptimizer:
    def __init__(self):
        self.df = None
        self.clusters = None
        self.simulation_results = None

    def generate_sample_data(self, n_points=150):
        """Gera dados simulados realistas para demonstração"""
        np.random.seed(42)
        
        # Coordenadas aproximadas de São Paulo e região metropolitana
        center_lat, center_lon = -23.5505, -46.6333
        
        # Criar 3-4 concentrações regionais
        concentrations = [
            (-23.5505, -46.6333, 40),  # Centro SP
            (-23.6821, -46.8755, 30),  # Osasco
            (-23.4538, -46.5333, 25),  # Guarulhos
            (-23.7937, -46.6816, 35),  # São Bernardo
            (-23.3200, -46.7311, 20),  # Jundiaí
        ]
        
        data = []
        seller_id = 1000
        
        for i, (lat_center, lon_center, n_stores) in enumerate(concentrations):
            for _ in range(n_stores):
                # Variação aleatória ao redor do centro
                lat = lat_center + np.random.normal(0, 0.05)  # ~5km de variação
                lon = lon_center + np.random.normal(0, 0.05)
                
                # Volume de pacotes com distribuição realista
                volume = max(1, int(np.random.lognormal(3, 1)))  # Média ~20, max ~200
                
                data.append({
                    'EXPECTPICKUP': pd.Timestamp.now().date(),
                    'CLIENTEseller_id': f'SELLER_{seller_id}',
                    'ADDRESS': f'Rua Exemplo {seller_id % 1000}',
                    'CITY': 'São Paulo' if i < 2 else ['Osasco', 'Guarulhos', 'São Bernardo', 'Jundiaí'][i-2],
                    'STATE': 'SP',
                    'ZIP_CODE': f'{8000 + i}{seller_id % 1000:03d}',
                    'AREA': f'Região {i+1}',
                    'ROTA': f'ROTA_{i+1}',
                    'VOLUME_TIKTOK_PICKUP': volume,
                    'ENDERECO_COMPLETO': f'Rua Exemplo {seller_id % 1000}, São Paulo, SP',
                    'LATITUDE': lat,
                    'LONGITUDE': lon
                })
                seller_id += 1
        
        self.df = pd.DataFrame(data)
        return self.df
    
    def load_data(self, df=None, filepath=None):
        """Carrega dados do usuário"""
        if df is not None:
            self.df = df.copy()
        elif filepath:
            self.df = pd.read_csv(filepath)
        else:
            self.df = self.generate_sample_data()
        
        # Limpeza básica
        self.df = self.df.dropna(subset=['LATITUDE', 'LONGITUDE', 'VOLUME_TIKTOK_PICKUP'])
        self.df = self.df[self.df['VOLUME_TIKTOK_PICKUP'] > 0]
        
        # Verificações de segurança
        if len(self.df) == 0:
            raise ValueError("Nenhum dado válido encontrado após limpeza. Verifique as colunas LATITUDE, LONGITUDE e VOLUME_TIKTOK_PICKUP")
        
        # Verificar se as coordenadas são válidas
        invalid_coords = (
            (self.df['LATITUDE'].abs() > 90) | 
            (self.df['LONGITUDE'].abs() > 180) |
            (self.df['LATITUDE'].isna()) |
            (self.df['LONGITUDE'].isna())
        )
        
        if invalid_coords.sum() > 0:
            print(f"⚠️  Removendo {invalid_coords.sum()} pontos com coordenadas inválidas")
            self.df = self.df[~invalid_coords]
        
        print(f"Dataset carregado: {len(self.df)} pontos de coleta")
        print(f"Coordenadas - Lat: {self.df['LATITUDE'].min():.4f} a {self.df['LATITUDE'].max():.4f}")
        print(f"Coordenadas - Lon: {self.df['LONGITUDE'].min():.4f} a {self.df['LONGITUDE'].max():.4f}")
        print(f"Volume - Min: {self.df['VOLUME_TIKTOK_PICKUP'].min()}, Max: {self.df['VOLUME_TIKTOK_PICKUP'].max()}")
        
        return self.df
    
    def exploratory_analysis(self):
        """Análise exploratória dos dados"""
        print("=== ANÁLISE EXPLORATÓRIA ===")
        print(f"Total de pontos: {len(self.df)}")
        print(f"Volume total de pacotes: {self.df['VOLUME_TIKTOK_PICKUP'].sum()}")
        print(f"Volume médio por ponto: {self.df['VOLUME_TIKTOK_PICKUP'].mean():.1f}")
        print(f"Volume mediano: {self.df['VOLUME_TIKTOK_PICKUP'].median():.1f}")
        print(f"Desvio padrão do volume: {self.df['VOLUME_TIKTOK_PICKUP'].std():.1f}")
        
        # Estatísticas geográficas
        lat_range = self.df['LATITUDE'].max() - self.df['LATITUDE'].min()
        lon_range = self.df['LONGITUDE'].max() - self.df['LONGITUDE'].min()
        print(f"Dispersão geográfica - Lat: {lat_range:.4f}°, Lon: {lon_range:.4f}°")
        
        return {
            'total_points': len(self.df),
            'total_volume': self.df['VOLUME_TIKTOK_PICKUP'].sum(),
            'avg_volume': self.df['VOLUME_TIKTOK_PICKUP'].mean(),
            'geographic_spread': (lat_range, lon_range)
        }
    
    def calculate_distance_matrix(self, sample_size=None):
        """Calcula matriz de distâncias geográficas"""
        df_sample = self.df.sample(n=min(sample_size or len(self.df), 200))  # Limita para performance
        
        n = len(df_sample)
        distance_matrix = np.zeros((n, n))
        
        coords = df_sample[['LATITUDE', 'LONGITUDE']].values
        
        for i in range(n):
            for j in range(i+1, n):
                dist = geodesic(coords[i], coords[j]).kilometers
                distance_matrix[i][j] = dist
                distance_matrix[j][i] = dist
        
        return distance_matrix, df_sample
    
    def find_optimal_clusters(self, max_clusters=12):
        """Encontra número ótimo de clusters usando método do cotovelo e silhueta"""
        X = self.df[['LATITUDE', 'LONGITUDE']].values
        weights = self.df['VOLUME_TIKTOK_PICKUP'].values
        
        # Normalização ponderada
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Adicionar peso como terceira dimensão (normalizado)
        weights_scaled = (weights - weights.min()) / (weights.max() - weights.min())
        X_weighted = np.column_stack([X_scaled, weights_scaled * 0.5])  # Peso reduzido para não dominar
        
        inertias = []
        silhouette_scores = []
        K_range = range(2, max_clusters + 1)
        
        for k in K_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            cluster_labels = kmeans.fit_predict(X_weighted)
            
            inertias.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X_weighted, cluster_labels))
        
        # Encontrar cotovelo
        deltas = np.diff(inertias)
        second_deltas = np.diff(deltas)
        elbow_idx = np.argmax(second_deltas) + 2  # +2 devido aos diffs
        optimal_k_elbow = K_range[elbow_idx] if elbow_idx < len(K_range) else K_range[-1]
        
        # Melhor silhueta
        optimal_k_silhouette = K_range[np.argmax(silhouette_scores)]
        
        # Compromisso entre os dois métodos
        optimal_k = int(np.mean([optimal_k_elbow, optimal_k_silhouette]))
        
        print(f"Número ótimo de clusters:")
        print(f"  - Método do cotovelo: {optimal_k_elbow}")
        print(f"  - Melhor silhueta: {optimal_k_silhouette}")
        print(f"  - Escolhido (média): {optimal_k}")
        
        return optimal_k, (K_range, inertias, silhouette_scores)
    
    def perform_clustering(self, n_clusters=None):
        """Executa clustering geográfico ponderado"""
        if n_clusters is None:
            n_clusters, _ = self.find_optimal_clusters()
        
        X = self.df[['LATITUDE', 'LONGITUDE']].values
        weights = self.df['VOLUME_TIKTOK_PICKUP'].values
        
        # Preparação dos dados
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        weights_scaled = (weights - weights.min()) / (weights.max() - weights.min())
        X_weighted = np.column_stack([X_scaled, weights_scaled * 0.3])
        
        # K-means principal
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
        cluster_labels = kmeans.fit_predict(X_weighted)
        
        # Adicionar clusters ao dataframe
        self.df['CLUSTER'] = cluster_labels
        
        # Calcular estatísticas por cluster
        cluster_stats = []
        for cluster_id in range(n_clusters):
            cluster_data = self.df[self.df['CLUSTER'] == cluster_id]
            
            # Centróide geográfico
            centroid_lat = cluster_data['LATITUDE'].mean()
            centroid_lon = cluster_data['LONGITUDE'].mean()
            
            # Raio máximo do cluster
            max_distance = 0
            for _, row in cluster_data.iterrows():
                dist = geodesic((centroid_lat, centroid_lon), 
                              (row['LATITUDE'], row['LONGITUDE'])).kilometers
                max_distance = max(max_distance, dist)
            
            stats = {
                'cluster_id': cluster_id,
                'n_points': len(cluster_data),
                'total_volume': cluster_data['VOLUME_TIKTOK_PICKUP'].sum(),
                'avg_volume_per_point': cluster_data['VOLUME_TIKTOK_PICKUP'].mean(),
                'centroid_lat': centroid_lat,
                'centroid_lon': centroid_lon,
                'max_radius_km': max_distance,
                'cities': cluster_data['CITY'].value_counts().to_dict()
            }
            cluster_stats.append(stats)
        
        self.clusters = pd.DataFrame(cluster_stats)
        
        print("\n=== RESULTADOS DO CLUSTERING ===")
        for _, cluster in self.clusters.iterrows():
            print(f"Cluster {cluster['cluster_id']}:")
            print(f"  - Pontos: {cluster['n_points']}")
            print(f"  - Volume total: {cluster['total_volume']}")
            print(f"  - Raio máximo: {cluster['max_radius_km']:.1f} km")
            print(f"  - Principais cidades: {list(cluster['cities'].keys())[:3]}")
            print()
        
        return self.clusters
    
    def monte_carlo_simulation(self, n_simulations=5000):
        """Simulação Monte Carlo para dimensionamento de frota"""
        if self.clusters is None:
            raise ValueError("Execute o clustering primeiro!")
        
        # Parâmetros da simulação
        vehicle_capacities = [300, 500, 1000]
        trips_per_day = [1, 2]
        
        # Parâmetros de tempo (em horas) - mais conservadores
        base_pickup_time = 0.25  # 15 min por ponto
        travel_time_per_km = 0.08  # 3 min por km
        working_hours_per_day = 8
        max_reasonable_time = working_hours_per_day * 0.8  # 80% do tempo máximo
        
        simulation_results = []
        
        for _, cluster in self.clusters.iterrows():
            cluster_id = cluster['cluster_id']
            total_volume = cluster['total_volume']
            n_points = cluster['n_points']
            max_radius = cluster['max_radius_km']
            
            print(f"Simulando Cluster {cluster_id} (Volume: {total_volume}, Pontos: {n_points})...")
            
            for capacity in vehicle_capacities:
                for trips in trips_per_day:
                    daily_capacity = capacity * trips
                    vehicles_needed = []
                    
                    for sim in range(n_simulations):
                        # Variações aleatórias mais conservadoras
                        volume_variation = np.random.uniform(0.9, 1.1)    # ±10%
                        time_variation = np.random.uniform(0.9, 1.1)     # ±10%
                        traffic_factor = np.random.uniform(1.0, 1.3)     # Trânsito mais moderado
                        
                        adjusted_volume = max(1, int(total_volume * volume_variation))
                        
                        # Tempo estimado por veículo
                        pickup_time = n_points * base_pickup_time * time_variation
                        travel_time = max_radius * 2 * travel_time_per_km * traffic_factor
                        total_time_per_trip = pickup_time + travel_time
                        
                        # Capacidade diária inicial
                        current_daily_capacity = daily_capacity
                        
                        # Verificar se cabe no dia de trabalho
                        total_daily_time = total_time_per_trip * trips
                        if total_daily_time > max_reasonable_time:
                            # Reduzir eficiência se não couber no tempo
                            efficiency_penalty = total_daily_time / max_reasonable_time
                            current_daily_capacity = max(10, int(current_daily_capacity / efficiency_penalty))
                        
                        # Garantir que daily_capacity nunca seja zero ou muito baixo
                        current_daily_capacity = max(5, current_daily_capacity)
                        
                        # Calcular veículos necessários
                        vehicles = max(1, int(np.ceil(adjusted_volume / current_daily_capacity)))
                        
                        # Limite máximo razoável de veículos por cluster
                        vehicles = min(vehicles, max(10, n_points))
                        
                        vehicles_needed.append(vehicles)
                    
                    # Estatísticas da simulação
                    result = {
                        'cluster_id': cluster_id,
                        'capacity': capacity,
                        'trips': trips,
                        'daily_capacity': capacity * trips,
                        'min_vehicles': int(np.percentile(vehicles_needed, 5)),
                        'median_vehicles': int(np.percentile(vehicles_needed, 50)),
                        'max_vehicles': int(np.percentile(vehicles_needed, 95)),
                        'mean_vehicles': np.mean(vehicles_needed),
                        'std_vehicles': np.std(vehicles_needed)
                    }
                    simulation_results.append(result)
        
        self.simulation_results = pd.DataFrame(simulation_results)
        print(f"✅ Simulação concluída para {len(self.clusters)} clusters")
        return self.simulation_results
    
    def generate_fleet_recommendations(self):
        """Gera recomendações finais de frota"""
        if self.simulation_results is None:
            raise ValueError("Execute a simulação Monte Carlo primeiro!")
        
        recommendations = []
        
        for cluster_id in self.clusters['cluster_id'].unique():
            cluster_sims = self.simulation_results[
                self.simulation_results['cluster_id'] == cluster_id
            ]
            
            # Cenários
            conservative = cluster_sims[
                (cluster_sims['capacity'] == 100) & (cluster_sims['trips'] == 1)
            ].iloc[0]
            
            moderate = cluster_sims[
                (cluster_sims['capacity'] == 150) & (cluster_sims['trips'] == 1)
            ].iloc[0]
            
            optimistic = cluster_sims[
                (cluster_sims['capacity'] == 300) & (cluster_sims['trips'] == 2)
            ].iloc[0]
            
            cluster_info = self.clusters[self.clusters['cluster_id'] == cluster_id].iloc[0]
            
            recommendation = {
                'cluster_id': cluster_id,
                'total_volume': cluster_info['total_volume'],
                'n_points': cluster_info['n_points'],
                'conservative_range': f"{int(conservative['min_vehicles'])}-{int(conservative['max_vehicles'])}",
                'moderate_range': f"{int(moderate['min_vehicles'])}-{int(moderate['max_vehicles'])}",
                'optimistic_range': f"{int(optimistic['min_vehicles'])}-{int(optimistic['max_vehicles'])}",
                'recommended_scenario': 'moderate',
                'recommended_vehicles': f"{int(moderate['min_vehicles'])}-{int(moderate['max_vehicles'])}",
                'main_cities': ', '.join(list(cluster_info['cities'].keys())[:3])
            }
            recommendations.append(recommendation)
        
        recommendations_df = pd.DataFrame(recommendations)
        
        print("\n=== RECOMENDAÇÕES DE FROTA ===")
        print(recommendations_df.to_string(index=False))
        
        return recommendations_df
    
    def create_interactive_map(self):
        """Cria mapa interativo com clusters"""
        if self.df is None or 'CLUSTER' not in self.df.columns:
            raise ValueError("Execute o clustering primeiro!")
        
        # Centro do mapa
        center_lat = self.df['LATITUDE'].mean()
        center_lon = self.df['LONGITUDE'].mean()
        
        # Criar mapa
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=10,
            tiles='OpenStreetMap'
        )
        
        # Cores para clusters
        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
                 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 
                 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen']
        
        # Adicionar pontos por cluster
        for cluster_id in sorted(self.df['CLUSTER'].unique()):
            cluster_data = self.df[self.df['CLUSTER'] == cluster_id]
            color = colors[cluster_id % len(colors)]
            
            for _, row in cluster_data.iterrows():
                folium.CircleMarker(
                    location=[row['LATITUDE'], row['LONGITUDE']],
                    radius=max(3, min(15, row['VOLUME_TIKTOK_PICKUP'] / 10)),
                    popup=f"""
                    <b>Cluster {cluster_id}</b><br>
                    Seller: {row['CLIENTEseller_id']}<br>
                    Volume: {row['VOLUME_TIKTOK_PICKUP']} pacotes<br>
                    Cidade: {row['CITY']}<br>
                    Endereço: {row['ADDRESS']}
                    """,
                    color=color,
                    fill=True,
                    fillOpacity=0.6
                ).add_to(m)
        
        # Adicionar centróides dos clusters
        for _, cluster in self.clusters.iterrows():
            folium.Marker(
                location=[cluster['centroid_lat'], cluster['centroid_lon']],
                popup=f"""
                <b>Centro do Cluster {cluster['cluster_id']}</b><br>
                Pontos: {cluster['n_points']}<br>
                Volume Total: {cluster['total_volume']}<br>
                Raio: {cluster['max_radius_km']:.1f} km
                """,
                icon=folium.Icon(color='black', icon='info-sign')
            ).add_to(m)
            
            # Círculo de cobertura
            folium.Circle(
                location=[cluster['centroid_lat'], cluster['centroid_lon']],
                radius=cluster['max_radius_km'] * 1000,  # metros
                color=colors[cluster['cluster_id'] % len(colors)],
                fill=False,
                opacity=0.3
            ).add_to(m)
        
        return m
    
    def run_complete_analysis(self, df=None, n_clusters=None):
        """Executa análise completa"""
        print("🚚 INICIANDO ANÁLISE DE OTIMIZAÇÃO LOGÍSTICA")
        print("=" * 50)
        
        # 1. Carregar dados
        self.load_data(df)
        
        # 2. Análise exploratória
        self.exploratory_analysis()
        
        # 3. Clustering
        print("\n📍 EXECUTANDO CLUSTERING...")
        self.perform_clustering(n_clusters)
        
        # 4. Simulação Monte Carlo
        print("\n🎲 EXECUTANDO SIMULAÇÃO MONTE CARLO...")
        self.monte_carlo_simulation()
        
        # 5. Recomendações
        print("\n📊 GERANDO RECOMENDAÇÕES...")
        recommendations = self.generate_fleet_recommendations()
        
        # 6. Mapa
        print("\n🗺️  CRIANDO MAPA INTERATIVO...")
        map_viz = self.create_interactive_map()
        
        return {
            'data': self.df,
            'clusters': self.clusters,
            'simulation_results': self.simulation_results,
            'recommendations': recommendations,
            'map': map_viz
        }

    def export_clustered_data(self, filename=None, add_timestamp=True):
        """Exporta o DataFrame com a coluna de clusters para Excel"""
        if self.df is None or 'CLUSTER' not in self.df.columns:
            raise ValueError("Execute o clustering primeiro!")
        
        # Definir nome do arquivo
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") if add_timestamp else ""
            filename = f"dados_com_clusters_{timestamp}.xlsx" if timestamp else "dados_com_clusters.xlsx"
        elif add_timestamp and not filename.endswith(('.xlsx', '.xls')):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{filename}_{timestamp}.xlsx"
        elif not filename.endswith(('.xlsx', '.xls')):
            filename = f"{filename}.xlsx"
        
        try:
            # Criar uma cópia organizada dos dados
            export_df = self.df.copy()
            
            # Reorganizar colunas para ter CLUSTER no início
            cols = ['CLUSTER'] + [col for col in export_df.columns if col != 'CLUSTER']
            export_df = export_df[cols]
            
            # Ordenar por cluster
            export_df = export_df.sort_values(['CLUSTER', 'VOLUME_TIKTOK_PICKUP'], ascending=[True, False])
            
            # Adicionar estatísticas por cluster
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                # Aba principal com dados
                export_df.to_excel(writer, sheet_name='Dados_com_Clusters', index=False)
                
                # Aba com resumo dos clusters
                if self.clusters is not None:
                    cluster_summary = self.clusters.copy()
                    cluster_summary.to_excel(writer, sheet_name='Resumo_Clusters', index=False)
                
                # Aba com estatísticas gerais
                stats_data = []
                for cluster_id in sorted(export_df['CLUSTER'].unique()):
                    cluster_data = export_df[export_df['CLUSTER'] == cluster_id]
                    stats_data.append({
                        'Cluster': cluster_id,
                        'Total_Pontos': len(cluster_data),
                        'Total_Volume': cluster_data['VOLUME_TIKTOK_PICKUP'].sum(),
                        'Volume_Medio': cluster_data['VOLUME_TIKTOK_PICKUP'].mean(),
                        'Principais_Cidades': ', '.join(cluster_data['CITY'].value_counts().head(3).index.tolist())
                    })
                
                stats_df = pd.DataFrame(stats_data)
                stats_df.to_excel(writer, sheet_name='Estatisticas_Clusters', index=False)
            
            print(f"✅ Dados com clusters exportados: {filename}")
            print(f"   📊 {len(export_df)} pontos distribuídos em {export_df['CLUSTER'].nunique()} clusters")
            return filename
            
        except Exception as e:
            print(f"❌ Erro ao exportar dados: {e}")
            return None

    def export_fleet_recommendations(self, filename=None, add_timestamp=True):
        """Exporta as recomendações de frota para Excel"""
        if self.simulation_results is None:
            raise ValueError("Execute a simulação Monte Carlo primeiro!")
        
        # Definir nome do arquivo
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") if add_timestamp else ""
            filename = f"recomendacoes_frota_{timestamp}.xlsx" if timestamp else "recomendacoes_frota.xlsx"
        elif add_timestamp and not filename.endswith(('.xlsx', '.xls')):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{filename}_{timestamp}.xlsx"
        elif not filename.endswith(('.xlsx', '.xls')):
            filename = f"{filename}.xlsx"
        
        try:
            # Gerar recomendações se ainda não foi feito
            recommendations = self.generate_fleet_recommendations()
            
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                # Aba 1: Recomendações resumidas
                recommendations.to_excel(writer, sheet_name='Recomendacoes_Resumo', index=False)
                
                # Aba 2: Resultados detalhados da simulação
                simulation_detailed = self.simulation_results.copy()
                simulation_detailed.to_excel(writer, sheet_name='Simulacao_Detalhada', index=False)
                
                # Aba 3: Resumo executivo
                total_volume = self.df['VOLUME_TIKTOK_PICKUP'].sum()
                total_points = len(self.df)
                n_clusters = self.df['CLUSTER'].nunique()
                
                executive_summary = pd.DataFrame([
                    {'Metrica': 'Total de Pontos de Coleta', 'Valor': total_points},
                    {'Metrica': 'Volume Total de Pacotes', 'Valor': total_volume},
                    {'Metrica': 'Número de Clusters', 'Valor': n_clusters},
                    {'Metrica': 'Volume Médio por Ponto', 'Valor': f"{total_volume/total_points:.1f}"},
                    {'Metrica': 'Data da Análise', 'Valor': datetime.now().strftime("%d/%m/%Y %H:%M")},
                    {'Metrica': 'Cenário Recomendado', 'Valor': 'Moderado (150 pacotes/veículo, 1 viagem/dia)'},
                ])
                executive_summary.to_excel(writer, sheet_name='Resumo_Executivo', index=False)
                
                # Aba 4: Análise por cluster
                cluster_analysis = []
                for _, rec in recommendations.iterrows():
                    cluster_info = self.clusters[self.clusters['cluster_id'] == rec['cluster_id']].iloc[0]
                    cluster_analysis.append({
                        'Cluster': rec['cluster_id'],
                        'Pontos': rec['n_points'],
                        'Volume_Total': rec['total_volume'],
                        'Cidades_Principais': rec['main_cities'],
                        'Raio_Cobertura_km': f"{cluster_info['max_radius_km']:.1f}",
                        'Veiculos_Conservador': rec['conservative_range'],
                        'Veiculos_Moderado': rec['moderate_range'],
                        'Veiculos_Otimista': rec['optimistic_range'],
                        'Recomendacao': rec['recommended_vehicles']
                    })
                
                cluster_analysis_df = pd.DataFrame(cluster_analysis)
                cluster_analysis_df.to_excel(writer, sheet_name='Analise_por_Cluster', index=False)
            
            print(f"✅ Recomendações de frota exportadas: {filename}")
            print(f"   🚛 Análise de {n_clusters} clusters com {total_volume} pacotes")
            return filename
            
        except Exception as e:
            print(f"❌ Erro ao exportar recomendações: {e}")
            return None

    def export_interactive_map(self, filename=None, add_timestamp=True, add_info_panel=True):
        """Exporta o mapa interativo para HTML"""
        if self.df is None or 'CLUSTER' not in self.df.columns:
            raise ValueError("Execute o clustering primeiro!")
        
        # Definir nome do arquivo
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") if add_timestamp else ""
            filename = f"mapa_clusters_{timestamp}.html" if timestamp else "mapa_clusters.html"
        elif add_timestamp and not filename.endswith('.html'):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"{filename}_{timestamp}.html"
        elif not filename.endswith('.html'):
            filename = f"{filename}.html"
        
        try:
            # Criar mapa base
            m = self.create_interactive_map()
            
            if add_info_panel:
                # Adicionar painel de informações
                total_volume = self.df['VOLUME_TIKTOK_PICKUP'].sum()
                total_points = len(self.df)
                n_clusters = self.df['CLUSTER'].nunique()
                
                info_html = f'''
                <div style="position: fixed; 
                            top: 10px; right: 10px; width: 320px; 
                            background-color: white; border: 2px solid #333; 
                            border-radius: 10px; z-index: 9999; 
                            font-family: Arial, sans-serif; font-size: 12px; 
                            padding: 15px; box-shadow: 0 4px 8px rgba(0,0,0,0.3);">
                <h3 style="margin-top: 0; color: #333; border-bottom: 2px solid #007cba; padding-bottom: 5px;">
                    📊 Análise de Clusters Logísticos
                </h3>
                <div style="margin: 10px 0;">
                    <p><b>📍 Total de Clusters:</b> {n_clusters}</p>
                    <p><b>📦 Pontos de Coleta:</b> {total_points}</p>
                    <p><b>📋 Volume Total:</b> {total_volume:,} pacotes</p>
                    <p><b>📊 Média por Ponto:</b> {total_volume/total_points:.1f} pacotes</p>
                    <p><b>📅 Data da Análise:</b> {datetime.now().strftime('%d/%m/%Y')}</p>
                </div>
                <hr style="border: 1px solid #ddd;">
                <div style="font-size: 10px; color: #666;">
                    <p><b>Legenda:</b></p>
                    <p>🔵 Pontos de coleta (tamanho = volume)</p>
                    <p>📍 Centróides dos clusters</p>
                    <p>⭕ Área de cobertura</p>
                </div>
                </div>
                '''
                m.get_root().html.add_child(folium.Element(info_html))
                
                # Adicionar título
                title_html = '''
                <h2 align="center" style="font-size: 24px; font-family: Arial, sans-serif; 
                                          color: #333; margin: 20px 0;">
                    🚚 Mapa de Clusters para Otimização Logística
                </h2>
                '''
                m.get_root().html.add_child(folium.Element(title_html))
            
            # Salvar mapa
            m.save(filename)
            
            print(f"✅ Mapa interativo exportado: {filename}")
            print(f"   🗺️  {n_clusters} clusters com {total_points} pontos")
            print(f"   💡 Abra o arquivo em qualquer navegador para visualizar")
            return filename
            
        except Exception as e:
            print(f"❌ Erro ao exportar mapa: {e}")
            return None

    def export_all_results(self, base_filename=None, add_timestamp=True):
        """Exporta todos os resultados (dados, recomendações e mapa)"""
        if self.df is None or 'CLUSTER' not in self.df.columns:
            raise ValueError("Execute a análise completa primeiro!")
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") if add_timestamp else ""
        
        print("📤 EXPORTANDO TODOS OS RESULTADOS...")
        print("=" * 40)
        
        results = {}
        
        # 1. Exportar dados com clusters
        print("1️⃣ Exportando dados com clusters...")
        data_file = base_filename + "_dados" if base_filename else "dados_com_clusters"
        results['data_file'] = self.export_clustered_data(data_file, add_timestamp)
        
        # 2. Exportar recomendações
        print("\n2️⃣ Exportando recomendações de frota...")
        rec_file = base_filename + "_recomendacoes" if base_filename else "recomendacoes_frota"
        results['recommendations_file'] = self.export_fleet_recommendations(rec_file, add_timestamp)
        
        # 3. Exportar mapa
        print("\n3️⃣ Exportando mapa interativo...")
        map_file = base_filename + "_mapa" if base_filename else "mapa_clusters"
        results['map_file'] = self.export_interactive_map(map_file, add_timestamp)
        
        print("\n✅ EXPORTAÇÃO CONCLUÍDA!")
        print(f"📁 Arquivos gerados:")
        for key, filename in results.items():
            if filename:
                print(f"   - {filename}")
        
        return results



In [102]:
# Inicializar
optimizer = LogisticsOptimizer()

# Executar análise completa com seus dados
results = optimizer.run_complete_analysis(df=df)

# OU testar com dados simulados primeiro
results = optimizer.run_complete_analysis()  

🚚 INICIANDO ANÁLISE DE OTIMIZAÇÃO LOGÍSTICA
Dataset carregado: 270 pontos de coleta
Coordenadas - Lat: -23.8746 a -23.3400
Coordenadas - Lon: -46.9232 a -46.4111
Volume - Min: 1, Max: 796
=== ANÁLISE EXPLORATÓRIA ===
Total de pontos: 270
Volume total de pacotes: 5083
Volume médio por ponto: 18.8
Volume mediano: 3.0
Desvio padrão do volume: 70.7
Dispersão geográfica - Lat: 0.5346°, Lon: 0.5121°

📍 EXECUTANDO CLUSTERING...
Número ótimo de clusters:
  - Método do cotovelo: 4
  - Melhor silhueta: 2
  - Escolhido (média): 3

=== RESULTADOS DO CLUSTERING ===
Cluster 0:
  - Pontos: 45
  - Volume total: 243
  - Raio máximo: 12.2 km
  - Principais cidades: ['São Paulo', 'Santo André']

Cluster 1:
  - Pontos: 46
  - Volume total: 293
  - Raio máximo: 26.1 km
  - Principais cidades: ['São Paulo', 'Cotia', 'Jandira']

Cluster 2:
  - Pontos: 179
  - Volume total: 4547
  - Raio máximo: 30.4 km
  - Principais cidades: ['São Paulo', 'Cajamar']


🎲 EXECUTANDO SIMULAÇÃO MONTE CARLO...
Simulando Cluster 

In [103]:
mapa = optimizer.create_interactive_map()

In [104]:
mapa

In [106]:
optimizer.export_interactive_map('mapa.html')

✅ Mapa interativo exportado: mapa.html
   🗺️  4 clusters com 150 pontos
   💡 Abra o arquivo em qualquer navegador para visualizar


'mapa.html'

In [107]:
optimizer.export_fleet_recommendations('resultado_recomendacao.xlsx')


=== RECOMENDAÇÕES DE FROTA ===
 cluster_id  total_volume  n_points conservative_range moderate_range optimistic_range recommended_scenario recommended_vehicles          main_cities
          0          2232        30              27-30          18-23             9-12             moderate                18-23            São Paulo
          1           636        21                7-8            5-6              3-3             moderate                  5-6 São Bernardo, Osasco
          2          2045        64              53-64          35-46            18-23             moderate                35-46    São Paulo, Osasco
          3          1234        35              18-23          12-15              6-8             moderate                12-15            Guarulhos
✅ Recomendações de frota exportadas: resultado_recomendacao.xlsx
   🚛 Análise de 4 clusters com 6147 pacotes


'resultado_recomendacao.xlsx'

In [112]:
optimizer.export_all_results('teste')

📤 EXPORTANDO TODOS OS RESULTADOS...
1️⃣ Exportando dados com clusters...
✅ Dados com clusters exportados: teste_dados_20250530_082003.xlsx
   📊 150 pontos distribuídos em 4 clusters

2️⃣ Exportando recomendações de frota...

=== RECOMENDAÇÕES DE FROTA ===
 cluster_id  total_volume  n_points conservative_range moderate_range optimistic_range recommended_scenario recommended_vehicles          main_cities
          0          2232        30              27-30          18-23             9-12             moderate                18-23            São Paulo
          1           636        21                7-8            5-6              3-3             moderate                  5-6 São Bernardo, Osasco
          2          2045        64              53-64          35-46            18-23             moderate                35-46    São Paulo, Osasco
          3          1234        35              18-23          12-15              6-8             moderate                12-15            Guar

{'data_file': 'teste_dados_20250530_082003.xlsx',
 'recommendations_file': 'teste_recomendacoes_20250530_082003.xlsx',
 'map_file': 'teste_mapa_20250530_082003.html'}

In [120]:
optimizer.run_complete_analysis(df=df)

🚚 INICIANDO ANÁLISE DE OTIMIZAÇÃO LOGÍSTICA
Dataset carregado: 270 pontos de coleta
Coordenadas - Lat: -23.8746 a -23.3400
Coordenadas - Lon: -46.9232 a -46.4111
Volume - Min: 1, Max: 796
=== ANÁLISE EXPLORATÓRIA ===
Total de pontos: 270
Volume total de pacotes: 5083
Volume médio por ponto: 18.8
Volume mediano: 3.0
Desvio padrão do volume: 70.7
Dispersão geográfica - Lat: 0.5346°, Lon: 0.5121°

📍 EXECUTANDO CLUSTERING...
Número ótimo de clusters:
  - Método do cotovelo: 4
  - Melhor silhueta: 2
  - Escolhido (média): 3

=== RESULTADOS DO CLUSTERING ===
Cluster 0:
  - Pontos: 45
  - Volume total: 243
  - Raio máximo: 12.2 km
  - Principais cidades: ['São Paulo', 'Santo André']

Cluster 1:
  - Pontos: 46
  - Volume total: 293
  - Raio máximo: 26.1 km
  - Principais cidades: ['São Paulo', 'Cotia', 'Jandira']

Cluster 2:
  - Pontos: 179
  - Volume total: 4547
  - Raio máximo: 30.4 km
  - Principais cidades: ['São Paulo', 'Cajamar']


🎲 EXECUTANDO SIMULAÇÃO MONTE CARLO...
Simulando Cluster 

{'data':     EXPECT PICKUP CLIENTE     CLIENTEseller_id  \
 0      2025-05-26  TIKTOK  7496179179138159616   
 1      2025-05-26  TIKTOK  7496161546569480192   
 2      2025-05-26  TIKTOK  7496147585319799808   
 3      2025-05-26  TIKTOK  7496162496353829888   
 4      2025-05-26  TIKTOK  7496159695603600384   
 ..            ...     ...                  ...   
 265    2025-05-26  TIKTOK  7496147577650910208   
 266    2025-05-26  TIKTOK  7496167854168179712   
 267    2025-05-26  TIKTOK  7496183141361420288   
 268    2025-05-26  TIKTOK  7496163512946820096   
 269    2025-05-26  TIKTOK  7496163471936100352   
 
                                                ADDRESS       CITY      STATE  \
 0                           RUA CAPITÃO FERRAIUOLO 367  São Paulo  São Paulo   
 1                      RUA DOUTOR ALCIDES DE CAMPOS 35  São Paulo  São Paulo   
 2    RUA CORONEL EMÍDIO PIEDADE 622 NO LADO ESQUERD...  São Paulo  São Paulo   
 3                                 RUA JOÃO BOEMER 125

In [118]:
optimizer.export_all_results('teste3')

📤 EXPORTANDO TODOS OS RESULTADOS...
1️⃣ Exportando dados com clusters...
✅ Dados com clusters exportados: teste3_dados_20250530_091149.xlsx
   📊 270 pontos distribuídos em 3 clusters

2️⃣ Exportando recomendações de frota...

=== RECOMENDAÇÕES DE FROTA ===
 cluster_id  total_volume  n_points conservative_range moderate_range optimistic_range recommended_scenario recommended_vehicles               main_cities
          0           243        45                5-6            3-4              2-2             moderate                  3-4    São Paulo, Santo André
          1           293        46                6-8            4-6              2-3             moderate                  4-6 São Paulo, Cotia, Jandira
          2          4547       179            179-179        179-179          101-132             moderate              179-179        São Paulo, Cajamar
✅ Recomendações de frota exportadas: teste3_recomendacoes_20250530_091149.xlsx
   🚛 Análise de 3 clusters com 5083 pacotes


{'data_file': 'teste3_dados_20250530_091149.xlsx',
 'recommendations_file': 'teste3_recomendacoes_20250530_091149.xlsx',
 'map_file': 'teste3_mapa_20250530_091149.html'}